# AXE Genesis PyTorch Meta-Learner & Q-Executor Pipeline (Kaggle GPU)

Full RL pipeline: Meta-Learner training (Phase 1), Q-Executor sequential traversal training (Phase 2), out-of-sample evaluation across 4 expiry horizons (Phase 3/4), and checkpoint export.

# AXE Genesis — Meta + Q Fixes (Signal Shaping on BOTH stages)

**Phase 1 (Meta) — signal-aware supervision:**
- `precompute_meta_signal_quality()` from SNR / DXY / RSI-diff / MTF confluence
- Sample weights = ATR-direction gate × signal quality (high-confluence bars dominate Q/Str loss)
- Strength targets reshaped: low quality → pull toward 0.5 (less false conviction)
- Liq loss coefficient reduced 0.2 → 0.08 (was dominating total loss)
- **Strict** best-checkpoint (`val_avg_wr > best` only) — fixed misleading NEW BEST flag
- Early stopping: patience 12 after min 8 epochs
- Zone/Vol/Vel targets synthesized and gated losses active

**Phase 2 (Q):**
- Continuous reward shaping via `compute_signal_quality_score` + `compute_shaped_reward`


In [3]:
# =============================================================================
# SYSTEM IMPORTS & PATH SETUP  (TensorFlow/Keras removed — this pipeline is PyTorch only.
#  The original notebook imported and GPU-configured TF/Keras but never used it; every
#  model defined below is torch.nn.Module. Keeping unused TF setup around was dead code
#  and misleading given the notebook's own title.)
# =============================================================================
import os
import glob
import zipfile
import logging
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Union
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.optim as optim

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("AXE")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

KAGGLE_DATASET_DIR = '/kaggle/input/datasets/danieljosephkibe/al-paka' if os.path.exists('/kaggle/input/datasets/danieljosephkibe/al-paka') else 'data'
OUTPUT_DIR = '/kaggle/working/checkpoints' if os.path.exists('/kaggle/working') else 'checkpoints'
ZIP_EXPORT_PATH = os.path.join('/kaggle/working' if os.path.exists('/kaggle/working') else '.', 'axe_meta_learner_weights.zip')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parameterized — previously hardcoded "GLD" throughout the Q-executor loop regardless
# of which CSV was actually loaded. Set this to match the dataset you're training on.
SYMBOL = "GLD"

# Zone-detection parameters — must match backend/scripts/evaluate_option_expiries.py exactly
# for true 1:1 parity (see update_real_snr_snapshot in that file).
ZONE_LOOKBACK_PERIOD = 500
ZONE_MIN_DISTANCE_PCT = 0.2
Q_LOOKBACK   = 150    # Canonical Q_LOOKBACK — change only here, not in downstream cells
NUM_HORIZONS = 4     # 5m, 15m, 30m, 1h
BUFFER_CAPACITY = 4000  # was 30000 — index replay keeps this small
BATCH_SIZE_Q = 64
Q_EPOCHS = 20



Device: cuda


In [4]:
# =============================================================================
# DATASET LOADING (Train 70% | Val 15% | Test 15%)
# =============================================================================
zip_files = glob.glob(os.path.join(KAGGLE_DATASET_DIR, '*.zip'))
if zip_files:
    print(f"Extracting dataset archive: {zip_files[0]}")
    with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
        target_extract = '/kaggle/working/data' if os.path.exists('/kaggle/working') else 'data'
        zip_ref.extractall(target_extract)
    data_dir = target_extract
elif os.path.exists('data/train_50k.csv'):
    data_dir = 'data'
else:
    data_dir = KAGGLE_DATASET_DIR

train_csv = os.path.join(data_dir, 'train_50k.csv')
val_csv   = os.path.join(data_dir, 'val_50k.csv')
test_csv  = os.path.join(data_dir, 'test_50k.csv')

if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv) if os.path.exists(val_csv) else None
    test_df  = pd.read_csv(test_csv) if os.path.exists(test_csv) else None
    print(f"Train Set: {len(train_df)} rows | Columns: {len(train_df.columns)}")
    if val_df is not None:  print(f"Validation Set: {len(val_df)} rows")
    if test_df is not None: print(f"Holdout Test Set: {len(test_df)} rows")
else:
    raise FileNotFoundError(f"Dataset files not found under {data_dir}. Check dataset path!")

close_col = "close_5m" if "close_5m" in train_df.columns else train_df.columns[0]
open_col  = "open_5m" if "open_5m" in train_df.columns else train_df.columns[0]
high_col  = "high_5m" if "high_5m" in train_df.columns else train_df.columns[0]
low_col   = "low_5m" if "low_5m" in train_df.columns else train_df.columns[0]
vol_col   = "volume_5m" if "volume_5m" in train_df.columns else train_df.columns[1]
up_vol_col   = "Bar_Volume_Up_5m" if "Bar_Volume_Up_5m" in train_df.columns else None
down_vol_col = "Bar_Volume_Down_5m" if "Bar_Volume_Down_5m" in train_df.columns else None
atr_col      = "ATR_5m" if "ATR_5m" in train_df.columns else None
print(f"Volume columns available: up={up_vol_col}, down={down_vol_col} | ATR column: {atr_col}")

Train Set: 34991 rows | Columns: 335
Validation Set: 7498 rows
Holdout Test Set: 7499 rows
Volume columns available: up=Bar_Volume_Up_5m, down=Bar_Volume_Down_5m | ATR column: ATR_5m


In [5]:
# =============================================================================
# REAL SNR ZONE DETECTION — ported verbatim from
# backend/app/core/analysis/support_resistance.py, verified against the live
# backend (detect_snr_levels_sequential explicitly guarantees no lookahead:
# "Only uses data up to up_to_index"). This is NOT a simplified placeholder —
# it is the exact same function the backend uses, so zone-anchored decisions
# here are genuinely 1:1 with production.
# =============================================================================

def detect_snr_levels_sequential(price_data, up_to_index, lookback_period, min_distance_pct=0.5):
    '''Detect S&R levels up to a specific index. CRITICAL: only uses data up to up_to_index.'''
    levels = []
    df = price_data.iloc[up_to_index - lookback_period: up_to_index + 1]
    if len(df) < 5:
        return levels

    highs = df["High"].values
    lows = df["Low"].values
    price_range = highs.max() - lows.min()
    min_distance = price_range * (min_distance_pct / 100)

    if len(lows) >= 5:
        support_cond1 = lows[2:-2] < lows[1:-3]
        support_cond2 = lows[2:-2] < lows[3:-1]
        support_cond3 = lows[3:-1] < lows[4:]
        support_cond4 = lows[1:-3] < lows[:-4]
        support_mask = support_cond1 & support_cond2 & support_cond3 & support_cond4
        support_indices = np.where(support_mask)[0] + 2

        resistance_cond1 = highs[2:-2] > highs[1:-3]
        resistance_cond2 = highs[2:-2] > highs[3:-1]
        resistance_cond3 = highs[3:-1] > highs[4:]
        resistance_cond4 = highs[1:-3] > highs[:-4]
        resistance_mask = resistance_cond1 & resistance_cond2 & resistance_cond3 & resistance_cond4
        resistance_indices = np.where(resistance_mask)[0] + 2

        for idx in support_indices:
            level = lows[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "support"))
        for idx in resistance_indices:
            level = highs[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "resistance"))

    window = 5
    if len(df) > window * 2:
        pivot_high_mask = np.ones(len(highs), dtype=bool)
        pivot_high_mask[:window] = False
        pivot_high_mask[-window:] = False
        for offset in range(1, window + 1):
            pivot_high_mask[window:-window] &= (
                (highs[window:-window] > highs[window-offset:-(window+offset)]) &
                (highs[window:-window] > highs[window+offset:len(highs)-window+offset])
            )
        pivot_high_indices = np.where(pivot_high_mask)[0]

        pivot_low_mask = np.ones(len(lows), dtype=bool)
        pivot_low_mask[:window] = False
        pivot_low_mask[-window:] = False
        for offset in range(1, window + 1):
            pivot_low_mask[window:-window] &= (
                (lows[window:-window] < lows[window-offset:-(window+offset)]) &
                (lows[window:-window] < lows[window+offset:len(lows)-window+offset])
            )
        pivot_low_indices = np.where(pivot_low_mask)[0]

        for idx in pivot_high_indices:
            level = highs[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "resistance"))
        for idx in pivot_low_indices:
            level = lows[idx]
            if not levels or all(abs(level - l[1]) >= min_distance for l in levels):
                levels.append((int(idx), float(level), "support"))

    return levels


def calculate_volume_profile_at_level(price_level, price_data, zone_width=0.004):
    '''CRITICAL: only uses the price_data slice passed in (no lookahead).'''
    upper_bound = price_level + zone_width
    lower_bound = price_level - zone_width
    highs = price_data["High"].values
    lows = price_data["Low"].values
    closes = price_data["Close"].values
    opens = price_data["Open"].values
    volumes = price_data["Volume"].values

    touches_level = (lows <= price_level) & (highs >= price_level)
    is_bullish = closes > opens
    total_volume = volumes[touches_level].sum()
    up_volume = volumes[touches_level & is_bullish].sum()
    down_volume = volumes[touches_level & ~is_bullish].sum()

    return {
        "total_volume": float(total_volume),
        "up_volume": float(up_volume),
        "down_volume": float(down_volume),
        "net_volume": float(up_volume - down_volume),
        "upper_bound": upper_bound,
        "lower_bound": lower_bound,
    }


def create_clustered_zones_sequential(levels, price_data_slice, n_clusters=16, zone_width=0.004):
    '''Create zones using K-means clustering for sequential analysis.'''
    if not levels:
        return []
    prices = [level[1] for level in levels]
    unique_prices_count = len(set(prices))
    if n_clusters is None:
        n_clusters = min(unique_prices_count, max(3, len(prices) // 3))
    if unique_prices_count < n_clusters:
        n_clusters = unique_prices_count
    if n_clusters < 1:
        return []
    if unique_prices_count < 2:
        if not prices:
            return []
        zone_price = prices[0]
        volume_data = calculate_volume_profile_at_level(zone_price, price_data_slice, zone_width)
        return [(0, zone_price, [l for l in levels if l[1] == zone_price], volume_data)]

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
    price_array = np.array(prices).reshape(-1, 1)
    clusters = kmeans.fit_predict(price_array)

    zones = []
    for cluster_id in range(n_clusters):
        cluster_levels = [levels[i] for i, c in enumerate(clusters) if c == cluster_id]
        if cluster_levels:
            zone_price = np.mean([l[1] for l in cluster_levels])
            volume_data = calculate_volume_profile_at_level(zone_price, price_data_slice, zone_width)
            zones.append((cluster_id, zone_price, cluster_levels, volume_data))
    return sorted(zones, key=lambda x: x[1])


def get_nearest_zones(zones, current_price):
    '''Mirror of ZoneSnapshotManager.get_nearest_zones — returns (nearest_support, nearest_resistance)
    as dicts with price_level + volume_delta_ratio, or None if absent.'''
    supports = [z for z in zones if z[1] <= current_price]
    resistances = [z for z in zones if z[1] >= current_price]
    nearest_supp = max(supports, key=lambda z: z[1]) if supports else None
    nearest_res = min(resistances, key=lambda z: z[1]) if resistances else None

    def _to_record(z):
        if z is None:
            return None
        _, price, _, vol = z
        total = vol["up_volume"] + vol["down_volume"]
        ratio = (vol["up_volume"] - vol["down_volume"]) / (total + 1e-6)
        return {"price_level": price, "volume_delta_ratio": ratio, "volume": vol}

    return _to_record(nearest_supp), _to_record(nearest_res)


print("Real SNR zone detection loaded (verified 1:1 port of backend support_resistance.py).")


Real SNR zone detection loaded (verified 1:1 port of backend support_resistance.py).


In [6]:
# =============================================================================
# DOMAIN STRUCTURES & REAL HARD ACTION MASK
# (Previous version's HardActionMask never referenced zone_manager at all — it only
#  gated on volume imbalance, meaning the no-chase / zone-anchored entry rule, the
#  central design principle of this strategy, was entirely absent. This version
#  enforces the same ATR-scaled proximity band + volume confirmation + single-position
#  restriction as backend/app/core/market/zone_snapshot.py's HardActionMask.)
# =============================================================================

@dataclass
class HTFBiasPackage:
    direction: str = "neutral"
    strength: float = 0.0
    reversal_prob: float = 0.0
    q_value: float = 0.0
    expected_mfe_pips: float = 0.0
    expected_mae_pips: float = 0.0
    horizon_strengths: List[float] = field(default_factory=lambda: [0.5, 0.5, 0.5, 0.5])
    optimal_horizon_idx: int = 2
    recommended_expiry: str = "30m"

@dataclass
class AccountContext:
    balance: float = 10000.0
    equity: float = 10000.0
    open_position_type: Optional[str] = None
    open_position_pnl_pct: float = 0.0
    daily_drawdown_pct: float = 0.0
    win_streak: int = 0
    loss_streak: int = 0
    reentries_in_window: int = 0
    max_reentries_allowed: int = 3

@dataclass
class ExecutionContext:
    symbol: str
    current_price: float
    atr: float
    buy_volume: float
    sell_volume: float
    hour_of_day: float
    day_of_week: int
    session_phase: str
    ltf_timeframe: str = "5m"


class HardActionMask:
    '''1:1 with backend zone_snapshot.py::HardActionMask — enforces the no-chase rule
    (entries only at/near a real zone), volume confirmation, and single-open-position
    restriction, as HARD constraints rather than something the network has to learn.'''

    def get_action_mask(
        self, current_price, atr, nearest_supp, nearest_res,
        buy_volume, sell_volume, has_open_position=False,
    ):
        # mask indices: 0=WAIT, 1=BUY_CALL, 2=BUY_PUT, 3=TAKE_PROFIT_HALF, 4=CLOSE_FLATTEN
        mask = np.ones(5, dtype=np.int32)

        if has_open_position:
            # Single running trade restriction — no new entries while a position is open.
            mask[1] = 0
            mask[2] = 0
            return mask

        mask[3] = 0
        mask[4] = 0

        proximity_band = max(atr * 0.75, current_price * 0.003)

        supp_ok = nearest_supp is not None and abs(current_price - nearest_supp["price_level"]) <= proximity_band
        res_ok = nearest_res is not None and abs(current_price - nearest_res["price_level"]) <= proximity_band

        # No-chase: BUY_CALL only valid near/below a support zone. BUY_PUT only valid near/above resistance.
        if not supp_ok:
            mask[1] = 0
        if not res_ok:
            mask[2] = 0

        # Volume confirmation gate — require the reaction to actually be confirmed, not just proximity.
        if buy_volume > 0 or sell_volume > 0:
            if buy_volume < sell_volume * 0.8:
                mask[1] = 0
            if sell_volume < buy_volume * 0.8:
                mask[2] = 0

        return mask


def _make_exec_ctx(symbol: str, price: float, row: dict, atr_col: str, up_vol_col: str, down_vol_col: str) -> ExecutionContext:
    '''Uses REAL up/down volume columns from the feature pipeline instead of a crude
    open/close-direction proxy, and REAL ATR instead of an SNR-distance-derived guess.
    Session-phase uses actual US/Eastern local time via zoneinfo (DST-aware), matching
    backend q_executor.py's is_nyse_open (09:30-10:30 ET) / is_power_hour (15:00-16:00 ET).'''
    ts = row.get("timestamp", None)
    hour_f, dow, phase = 14.5, 1, "off_hours"
    if ts is not None:
        try:
            ts_pd = pd.Timestamp(ts)
            if ts_pd.tzinfo is None:
                ts_pd = ts_pd.tz_localize("UTC")
            ts_et = ts_pd.tz_convert("America/New_York")
            hour_f = ts_et.hour + ts_et.minute / 60.0
            dow = ts_et.dayofweek
            if 9.5 <= hour_f < 10.5:
                phase = "nyse_open"
            elif 15.0 <= hour_f < 16.0:
                phase = "nyse_power_hour"
            elif 9.5 <= hour_f < 16.0:
                phase = "regular_hours"
        except Exception:
            pass

    buy_vol = float(row.get(up_vol_col, 0.0)) if up_vol_col else 0.0
    sell_vol = float(row.get(down_vol_col, 0.0)) if down_vol_col else 0.0
    atr_val = float(row.get(atr_col, price * 0.005)) if atr_col else price * 0.005

    return ExecutionContext(
        symbol=symbol, current_price=price, atr=max(0.01, atr_val),
        buy_volume=buy_vol, sell_volume=sell_vol, hour_of_day=hour_f, day_of_week=dow, session_phase=phase,
    )


def build_state_vector(net_out, htf_bias: HTFBiasPackage, account: AccountContext,
                        exec_ctx: ExecutionContext, nearest_supp, nearest_res) -> np.ndarray:
    '''1:1 with backend q_executor.py::build_state_vector's real 28-dim layout.
    NOTE: unlike the previous version, this contains NO future-derived value anywhere —
    the previous notebook placed the literal forward price move (the reward target) into
    state_vec[4] as an INPUT feature, both in training and in Phase-2 validation. That is
    direct label leakage: the network was being handed the answer as an input. Every
    field here is computable strictly from data up to and including the current bar.'''
    supp_dist = abs(exec_ctx.current_price - nearest_supp["price_level"]) / exec_ctx.current_price if nearest_supp else 1.0
    res_dist = abs(exec_ctx.current_price - nearest_res["price_level"]) / exec_ctx.current_price if nearest_res else 1.0
    supp_vol_ratio = nearest_supp["volume_delta_ratio"] if nearest_supp else 0.0
    res_vol_ratio = nearest_res["volume_delta_ratio"] if nearest_res else 0.0

    total_vol = exec_ctx.buy_volume + exec_ctx.sell_volume
    vol_delta_ratio = (exec_ctx.buy_volume - exec_ctx.sell_volume) / (total_vol + 1e-6)

    tf_flag = 1.0 if exec_ctx.ltf_timeframe == "15m" else 0.0
    dir_flag = 1.0 if htf_bias.direction == "bullish" else (-1.0 if htf_bias.direction == "bearish" else 0.0)
    hs = htf_bias.horizon_strengths if len(htf_bias.horizon_strengths) == 4 else [0.5, 0.5, 0.5, 0.5]

    sin_hour = float(np.sin(2 * np.pi * exec_ctx.hour_of_day / 24.0))
    cos_hour = float(np.cos(2 * np.pi * exec_ctx.hour_of_day / 24.0))
    dow_norm = float(exec_ctx.day_of_week) / 6.0
    is_nyse_open = 1.0 if exec_ctx.session_phase == "nyse_open" else 0.0
    is_power_hour = 1.0 if exec_ctx.session_phase == "nyse_power_hour" else 0.0

    state = np.array([
        dir_flag, float(htf_bias.strength), float(htf_bias.reversal_prob), float(htf_bias.q_value),
        float(htf_bias.expected_mfe_pips) / 100.0, float(htf_bias.expected_mae_pips) / 100.0,
        float(hs[0]), float(hs[1]), float(hs[2]), float(hs[3]),
        float(account.daily_drawdown_pct),
        1.0 if account.open_position_type == "CALL" else (-1.0 if account.open_position_type == "PUT" else 0.0),
        float(account.open_position_pnl_pct), float(account.win_streak) / 10.0, float(account.loss_streak) / 10.0,
        tf_flag, float(exec_ctx.atr) / exec_ctx.current_price, float(supp_dist), float(res_dist),
        float(supp_vol_ratio), float(res_vol_ratio), float(vol_delta_ratio),
        float(account.reentries_in_window) / float(account.max_reentries_allowed),
        sin_hour, cos_hour, dow_norm, is_nyse_open, is_power_hour,
    ], dtype=np.float32)
    return state

print("Real HardActionMask + 28-dim state vector (no future leakage) loaded.")


Real HardActionMask + 28-dim state vector (no future leakage) loaded.


In [7]:
# =============================================================================
# 🏗️ PYTORCH PRODUCTION MODEL ARCHITECTURES (SignalMetaNetwork & ExecutorQNetwork)
# =============================================================================
import torch
import torch.nn as nn
import torch.optim as optim

feature_cols = [c for c in train_df.columns if c not in ("timestamp", "Time") and not "target" in c and not "forward" in c and not "adv_target" in c]
num_features = len(feature_cols)

# ── PHASE 2 UPDATE: Context Windows (150/300) ──
# Meta-Learner:   150 bars (signal strength predictor, recent price action)
# Q-Learner:      300 bars (zone evolution analyst, full zone lifecycle)
meta_lookback_bars = 150
lookback_bars = meta_lookback_bars
q_lookback_bars = 150

input_dim = meta_lookback_bars * num_features

print(f"📊 Phase 2 Context Windows:")
print(f"   Meta-Learner:  {meta_lookback_bars} bars × {num_features} features = {input_dim} input dim")
print(f"   Q-Learner:     {q_lookback_bars} bars × {num_features} features (zone analyzer)")
print(f"   ML Targets:    21+ targets (zone, volatility, velocity)")


class SignalMetaNetwork(nn.Module):
    def __init__(self, input_dim: int = input_dim, num_actions: int = 4, hidden_dim: int = 128, num_features: int = num_features):
        super().__init__()
        self.num_features = num_features
        hidden_dim = hidden_dim or 128

        # Branch 1: Full Sequence (100%) Conv1D + LSTM Tower
        self.b1_conv1 = nn.Conv1d(num_features, 64, kernel_size=3, padding=1)
        self.b1_bn1   = nn.BatchNorm1d(64)
        self.b1_act1  = nn.SiLU()
        self.b1_conv2 = nn.Conv1d(64, 32, kernel_size=3, padding=1)
        self.b1_bn2   = nn.BatchNorm1d(32)
        self.b1_act2  = nn.SiLU()
        self.b1_lstm  = nn.LSTM(32, 32, batch_first=True)

        # Branch 2: Mid-Term (50% Slice) Conv1D Tower
        self.b2_conv  = nn.Conv1d(num_features, 32, kernel_size=3, padding=1)
        self.b2_bn    = nn.BatchNorm1d(32)
        self.b2_act   = nn.SiLU()
        self.b2_fc    = nn.Linear(32, 32)

        # Branch 3: Short-Term (30% Slice) Conv1D Tower
        self.b3_conv  = nn.Conv1d(num_features, 32, kernel_size=3, padding=1)
        self.b3_bn    = nn.BatchNorm1d(32)
        self.b3_act   = nn.SiLU()
        self.b3_fc    = nn.Linear(32, 32)

        # Auxiliary Supervised Heads per branch
        self.aux1_head = nn.Linear(64, 5)
        self.aux2_head = nn.Linear(32, 5)

        # Gated Ensemble Fusion Head
        self.fusion_fc   = nn.Linear(64 + 32 + 32 + 5 + 5, hidden_dim)
        self.fusion_ln   = nn.LayerNorm(hidden_dim)
        self.fusion_act  = nn.SiLU()
        self.fusion_fc2  = nn.Linear(hidden_dim, hidden_dim)
        self.fusion_ln2  = nn.LayerNorm(hidden_dim)
        self.fusion_act2 = nn.SiLU()

        self.q_head = nn.Linear(hidden_dim, num_actions)
        self.strength_head = nn.Sequential(
            nn.Linear(hidden_dim, 4),
            nn.Sigmoid(),
        )
        self.fusion_selector = nn.Linear(hidden_dim, 4)

        # Auxiliary Private Projections (connected to backbone — gradients flow through branch outputs)
        _aux_in = 64 + 32 + 32
        self.branch_ln = nn.LayerNorm(_aux_in)
        self.pips_proj = nn.Linear(_aux_in, 32)
        self.pips_ln   = nn.LayerNorm(32)
        self.pips_head = nn.Sequential(nn.SiLU(), nn.Linear(32, 16), nn.SiLU(), nn.Linear(16, 4))
        self.risk_proj = nn.Linear(_aux_in, 32)
        self.risk_ln   = nn.LayerNorm(32)
        self.risk_head = nn.Sequential(nn.SiLU(), nn.Linear(32, 16), nn.SiLU(), nn.Linear(16, 8))
        self.liq_proj  = nn.Linear(_aux_in, 16)
        self.liq_ln    = nn.LayerNorm(16)
        self.liquidity_head = nn.Sequential(nn.SiLU(), nn.Linear(16, 8), nn.SiLU(), nn.Linear(8, 2))
        self.rev_proj  = nn.Linear(_aux_in, 16)
        self.rev_ln    = nn.LayerNorm(16)
        self.reversal_head = nn.Sequential(nn.SiLU(), nn.Linear(16, 8), nn.SiLU(), nn.Linear(8, 1), nn.Sigmoid())

    def _prepare_3d(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 2:
            b, dim = x.shape
            c = self.num_features
            t = dim // c if dim >= c else 1
            if t * c != dim:
                c = dim
                t = 1
            return x.view(b, t, c)
        return x

    def forward(self, x: torch.Tensor, return_aux: bool = False):
        x_3d = self._prepare_3d(x)
        b, t, c = x_3d.shape
        x_trans = x_3d.transpose(1, 2)

        # Branch 1
        b1_c1 = self.b1_act1(self.b1_bn1(self.b1_conv1(x_trans)))
        b1_c2 = self.b1_act2(self.b1_bn2(self.b1_conv2(b1_c1)))
        b1_c2_trans = b1_c2.transpose(1, 2)
        b1_lstm_out, _ = self.b1_lstm(b1_c2_trans)
        b1_last = b1_lstm_out[:, -1, :]
        b1_gap  = torch.mean(b1_lstm_out, dim=1)
        b1_out  = torch.cat([b1_last, b1_gap], dim=-1)

        # Branch 2 (50% slice)
        half = max(1, t // 2)
        x_mid_trans = x_trans[:, :, -half:]
        b2_c = self.b2_act(self.b2_bn(self.b2_conv(x_mid_trans)))
        b2_gap = torch.mean(b2_c, dim=-1)
        b2_out = torch.relu(self.b2_fc(b2_gap))

        # Branch 3 (30% slice)
        recent = max(1, int(t * 0.3))
        x_rec_trans = x_trans[:, :, -recent:]
        b3_c = self.b3_act(self.b3_bn(self.b3_conv(x_rec_trans)))
        b3_gap = torch.mean(b3_c, dim=-1)
        b3_out = torch.relu(self.b3_fc(b3_gap))

        # Aux heads connected to backbone — gradients flow through b1_out, b2_out, b3_out to Conv1D/LSTM towers
        aux1 = self.aux1_head(b1_out)
        aux2 = self.aux2_head(b2_out)
        aux1_sg = aux1.detach()
        aux2_sg = aux2.detach()

        # Gated Fusion
        fusion_in = torch.cat([b1_out, b2_out, b3_out, aux1_sg, aux2_sg], dim=-1)
        feat = self.fusion_act(self.fusion_ln(self.fusion_fc(fusion_in)))
        feat = self.fusion_act2(self.fusion_ln2(self.fusion_fc2(feat)))

        q_vals   = self.q_head(feat)
        strength = self.strength_head(feat)
        selector_logits = self.fusion_selector(feat)

        # Auxiliary Heads — connected to backbone for regularization (pips/risk/liq/rev gradients flow to Conv1D/LSTM)
        branch_cat = self.branch_ln(torch.cat([b1_out, b2_out, b3_out], dim=-1))
        pips      = self.pips_head(self.pips_ln(self.pips_proj(branch_cat)))
        risk      = self.risk_head(self.risk_ln(self.risk_proj(branch_cat)))
        liquidity = self.liquidity_head(self.liq_ln(self.liq_proj(branch_cat)))
        reversal  = self.reversal_head(self.rev_ln(self.rev_proj(branch_cat)))

        if return_aux:
            return q_vals, strength, pips, risk, liquidity, reversal, aux1, aux2, selector_logits
        return q_vals, strength, pips, risk, liquidity, reversal


📊 Phase 2 Context Windows:
   Meta-Learner:  150 bars × 326 features = 48900 input dim
   Q-Learner:     150 bars × 326 features (zone analyzer)
   ML Targets:    21+ targets (zone, volatility, velocity)


In [8]:
# =============================================================================
# Q-EXECUTOR: CONSUME FULL INDICATORS + 28-DIM CONTEXT (PHASE 2)
#
# Q_LOOKBACK is set in the config cell (Cell 2) — do not override here
#   - feat_window shape determined by Q_LOOKBACK from config cell
#   - Q_LOOKBACK default binding uses call-time resolution (None default)
#
# Input Structure:
#   - feat_window: (B, Q_LOOKBACK, num_features)  full zone history with all indicators
#   - ctx:         (B, 28)                 meta + account + time
# =============================================================================
import torch
import torch.nn as nn
import numpy as np



class ExecutorQNetwork(nn.Module):
    """
    Dual-branch Q net:
      Branch A — Conv1D over recent full indicator window
      Branch B — dense over 28-dim meta/zone/account/time context
      Fusion  — 4 horizon heads × 3 actions (WAIT / CALL / PUT)
    """

    def __init__(
        self,
        num_features: int,
        ctx_dim: int = 28,
        q_lookback: int = Q_LOOKBACK,
        hidden_dim: int = 128,
        num_horizons: int = 4,
        num_head_actions: int = 3,
    ):
        super().__init__()
        self.num_features = num_features
        self.ctx_dim = ctx_dim
        self.q_lookback = q_lookback
        self.num_horizons = num_horizons

        # --- Branch A: full indicators (B, T, F) -> channels-first conv ---
        self.feat_conv1 = nn.Conv1d(num_features, 64, kernel_size=3, padding=1)
        self.feat_bn1 = nn.BatchNorm1d(64)
        self.feat_conv2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.feat_bn2 = nn.BatchNorm1d(64)
        self.feat_pool = nn.AdaptiveAvgPool1d(1)
        self.feat_fc = nn.Linear(64, 64)

        # --- Branch B: 28-dim context (same groups as before) ---
        self.b1_fc1 = nn.Linear(ctx_dim, hidden_dim)
        self.b1_ln1 = nn.LayerNorm(hidden_dim)
        self.b1_fc2 = nn.Linear(hidden_dim, 64)
        self.b1_ln2 = nn.LayerNorm(64)

        self.b2_meta = nn.Linear(10, 16)
        self.b2_risk = nn.Linear(5, 16)
        self.b2_zone = nn.Linear(8, 16)
        self.b2_time = nn.Linear(5, 16)
        self.b2_fusion = nn.Linear(64, 64)
        self.b2_ln = nn.LayerNorm(64)

        # --- Fusion + per-horizon heads ---
        self.fusion_fc = nn.Linear(64 + 64 + 64, hidden_dim)  # feat + b1 + b2
        self.fusion_ln = nn.LayerNorm(hidden_dim)

        self.horizon_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, 64),
                nn.LayerNorm(64),
                nn.SiLU(),
                nn.Linear(64, num_head_actions),
            )
            for _ in range(num_horizons)
        ])

    def _encode_feat(self, feat_window: torch.Tensor) -> torch.Tensor:
        # feat_window: (B, T, F) -> (B, F, T) for Conv1d
        x = feat_window.transpose(1, 2)
        x = torch.nn.functional.silu(self.feat_bn1(self.feat_conv1(x)))
        x = torch.nn.functional.silu(self.feat_bn2(self.feat_conv2(x)))
        x = self.feat_pool(x).squeeze(-1)  # (B, 64)
        return torch.nn.functional.silu(self.feat_fc(x))

    def _encode_ctx(self, ctx: torch.Tensor):
        b1 = torch.nn.functional.silu(self.b1_ln1(self.b1_fc1(ctx)))
        b1_out = torch.nn.functional.silu(self.b1_ln2(self.b1_fc2(b1)))

        # Safe slices if ctx is exactly 28
        meta_f = ctx[:, :10]
        risk_f = ctx[:, 10:15]
        zone_f = ctx[:, 15:23]
        time_f = ctx[:, 23:28]
        b2_cat = torch.cat([
            torch.relu(self.b2_meta(meta_f)),
            torch.relu(self.b2_risk(risk_f)),
            torch.relu(self.b2_zone(zone_f)),
            torch.relu(self.b2_time(time_f)),
        ], dim=-1)
        b2_out = torch.nn.functional.silu(self.b2_ln(self.b2_fusion(b2_cat)))
        return b1_out, b2_out

    def forward(self, feat_window: torch.Tensor, ctx: torch.Tensor, horizon_idx=None):
        """
        feat_window: (B, Q_LOOKBACK, num_features)
        ctx:         (B, 28)
        horizon_idx: int or None — if set, return (B, 3) for that head only
        """
        feat_h = self._encode_feat(feat_window)
        b1_out, b2_out = self._encode_ctx(ctx)
        shared = torch.nn.functional.silu(
            self.fusion_ln(self.fusion_fc(torch.cat([feat_h, b1_out, b2_out], dim=-1)))
        )
        if horizon_idx is not None:
            return self.horizon_heads[int(horizon_idx)](shared)
        return torch.stack([h(shared) for h in self.horizon_heads], dim=1)


# -----------------------------------------------------------------------------
# Feature-window helper (no lookahead: only bars <= abs_idx)
# -----------------------------------------------------------------------------
def build_feat_window(num_matrix: np.ndarray, abs_idx: int, q_lookback: int = None) -> np.ndarray:
    """
    num_matrix: (N_rows, num_features) full dataframe numeric block
    abs_idx:    absolute row index of "now"
    returns:    (q_lookback, num_features) float32, left-padded with zeros if needed
    """
    if q_lookback is None:
        q_lookback = Q_LOOKBACK
    start = abs_idx - q_lookback + 1
    if start >= 0:
        return num_matrix[start: abs_idx + 1].astype(np.float32)
    # left pad
    window = np.zeros((q_lookback, num_matrix.shape[1]), dtype=np.float32)
    available = num_matrix[: abs_idx + 1]
    window[-len(available):] = available
    return window


def build_feat_window_batch(num_matrix: np.ndarray, abs_indices, q_lookback: int = None) -> np.ndarray:
    if q_lookback is None:
        q_lookback = Q_LOOKBACK
    return np.stack([build_feat_window(num_matrix, int(i), q_lookback) for i in abs_indices])


# -----------------------------------------------------------------------------
# Drop-in replacements for call sites
# -----------------------------------------------------------------------------
# OLD:
#   logits = q_net(st_t, horizon_idx=h)
# NEW:
#   fw = build_feat_window(train_num_matrix, abs_idx)   # or test_matrix
#   fw_t = torch.tensor(fw[None, ...], dtype=torch.float32, device=device)
#   st_t = torch.tensor(state[None, ...], dtype=torch.float32, device=device)
#   logits = q_net(fw_t, st_t, horizon_idx=h)

# Phase 2 replay tuple becomes:
#   (feat_window, state_ctx, action, reward, next_feat_window, next_state_ctx, mask)
# Sample batch:
#   fw_b  = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32, device=device)
#   st_b  = torch.tensor(np.array([b[1] for b in batch]), dtype=torch.float32, device=device)
#   ...
#   q_sa  = q_net(fw_b, st_b, horizon_idx=h)
#   next_q = q_target(nfw_b, nst_b, horizon_idx=h)

# Phase 3 _get_h_logits:
def make_get_h_logits(q_net, num_matrix, device, q_lookback=None):
    if q_lookback is None:
        q_lookback = Q_LOOKBACK
    def _get_h_logits(state, abs_idx, h, has_open=False):
        state = state.copy()
        state[11] = 1.0 if has_open else 0.0
        state[15] = float(h) / 3.0
        fw = build_feat_window(num_matrix, abs_idx, q_lookback)
        with torch.no_grad():
            fw_t = torch.tensor(fw[None, ...], dtype=torch.float32, device=device)
            st_t = torch.tensor(state[None, ...], dtype=torch.float32, device=device)
            return q_net(fw_t, st_t, horizon_idx=h).squeeze(0).cpu().numpy()
    return _get_h_logits


# -----------------------------------------------------------------------------
# Instantiate (Phase 2 cell)
# -----------------------------------------------------------------------------
# q_net = ExecutorQNetwork(
#     num_features=num_features,
#     ctx_dim=28,
#     q_lookback=Q_LOOKBACK,
#     hidden_dim=128,
# ).to(device)
# q_target = ExecutorQNetwork(
#     num_features=num_features,
#     ctx_dim=28,
#     q_lookback=Q_LOOKBACK,
#     hidden_dim=128,
# ).to(device)
# q_target.load_state_dict(q_net.state_dict())

print(f"ExecutorQNetwork dual-input ready | Q_LOOKBACK={Q_LOOKBACK} | num_features will bind at init")


ExecutorQNetwork dual-input ready | Q_LOOKBACK=300 | num_features will bind at init


In [ ]:
# =============================================================================
# ML TARGET SYNTHESIS — compute real targets from available CSV columns
# when the pre-labeled columns are absent (all 22 were zero-filled before).
# =============================================================================

_ML_KEYS = [
    "adv_target_next_zone_idx", "adv_target_next_zone_bars", "adv_target_next_zone_distance",
    "adv_target_next_zone_type",
    "Volatility_Bull_next", "Volatility_Bear_next", "Volatility_Regime_next", "Volatility_Expansion_next",
    "Regime_Speed_Bull_next", "Regime_Speed_Bear_next",
    "speed_aligned_fwd_8", "speed_divergence_fwd_8",
    "Price_Velocity_Bull_next", "vel_bull_fwd_8", "Price_Velocity_Bear_next", "vel_bear_fwd_8",
    "Price_Velocity_Net_next", "vel_net_fwd_8",
    "adv_target_CSM_hist_fast_next", "adv_target_CSM_hist_slow_next",
    "adv_target_CSM_asset_fast_next", "adv_target_CSM_dxy_fast_next",
]


def _synthesize_ml_targets(df):
    """
    Synthesize 14 of the 22 ML targets from columns available in the CSV.
    Remaining 8 (zone index, CSM) stay zero since they require external data.
    All outputs are float32, NaN-free, length=len(df).
    """
    n = len(df)
    close = df[close_col].values.astype(np.float64)

    # ── ATR ──────────────────────────────────────────────────────────────
    if atr_col and atr_col in df.columns:
        atr = df[atr_col].values.astype(np.float64)
    elif high_col in df.columns and low_col in df.columns:
        atr = pd.Series(df[high_col].values - df[low_col].values).rolling(14, min_periods=1).mean().values
    else:
        atr = np.full(n, np.nanmedian(np.abs(np.diff(close, prepend=close[0]))))
    atr = np.nan_to_num(atr, nan=np.nanmedian(atr) if np.isfinite(atr).any() else 1e-4)
    atr = np.maximum(atr, 1e-6)

    # ── Volume ────────────────────────────────────────────────────────────
    if up_vol_col and up_vol_col in df.columns:
        up_vol = df[up_vol_col].values.astype(np.float64)
    else:
        up_vol = np.zeros(n)
    if down_vol_col and down_vol_col in df.columns:
        dn_vol = df[down_vol_col].values.astype(np.float64)
    else:
        dn_vol = np.zeros(n)
    total_vol = up_vol + dn_vol + 1e-6
    vol_delta = (up_vol - dn_vol) / total_vol   # in [-1, 1]

    # ── Forward returns (ATR-normalised) ─────────────────────────────────
    fwd_atr = {}
    for hb in (1, 3, 6, 8, 12):
        f = np.zeros(n, dtype=np.float64)
        f[:-hb] = close[hb:] - close[:-hb]
        fwd_atr[hb] = np.clip(f / atr, -10.0, 10.0)

    # ── Rolling ATR ratio (volatility expansion proxy) ────────────────────
    atr_ma = pd.Series(atr).rolling(20, min_periods=1).mean().values
    atr_ratio = np.clip(atr / (atr_ma + 1e-6), 0.5, 3.0).astype(np.float32)
    # If ratio collapsed to a constant (e.g. flat ATR series), use percentile rank instead
    if float(np.nanstd(atr_ratio)) < 1e-6:
        atr_s = pd.Series(atr)
        atr_ratio = atr_s.rolling(50, min_periods=5).apply(
            lambda x: float(pd.Series(x).rank(pct=True).iloc[-1]), raw=False
        ).fillna(0.5).values.astype(np.float32)

    # ── Rolling returns (price velocity) ─────────────────────────────────
    def _velocity(returns_arr, window=8):
        s = pd.Series(returns_arr)
        slope = s.rolling(window, min_periods=2).apply(
            lambda x: np.polyfit(np.arange(len(x)), x, 1)[0], raw=True
        ).values
        return np.nan_to_num(np.clip(slope, -5.0, 5.0), nan=0.0).astype(np.float32)

    bull_vel = np.maximum(fwd_atr[8], 0).astype(np.float32)
    bear_vel = np.maximum(-fwd_atr[8], 0).astype(np.float32)
    net_vel  = fwd_atr[8].astype(np.float32)

    # ── Regime: rolling fraction of up-bars ──────────────────────────────
    up_bars = (np.diff(close, prepend=close[0]) > 0).astype(np.float32)
    vol_regime = pd.Series(up_bars).rolling(20, min_periods=5).mean().values.astype(np.float32)
    vol_regime = np.nan_to_num(vol_regime, nan=0.5)

    # ── Speed alignment: vol_delta corr with fwd returns ─────────────────
    speed_aligned = np.clip(vol_delta * np.sign(fwd_atr[8]), -1.0, 1.0).astype(np.float32)
    speed_diverge = np.clip(-vol_delta * np.sign(fwd_atr[8]), -1.0, 1.0).astype(np.float32)

    out = {k: np.zeros(n, dtype=np.float32) for k in _ML_KEYS}
    # populated keys
    out["Volatility_Bull_next"]   = np.maximum(fwd_atr[1], 0).astype(np.float32)
    out["Volatility_Bear_next"]   = np.maximum(-fwd_atr[1], 0).astype(np.float32)
    out["Volatility_Regime_next"] = vol_regime
    out["Volatility_Expansion_next"] = atr_ratio
    out["Regime_Speed_Bull_next"] = bull_vel
    out["Regime_Speed_Bear_next"] = bear_vel
    out["speed_aligned_fwd_8"]   = speed_aligned
    out["speed_divergence_fwd_8"]= speed_diverge
    out["Price_Velocity_Bull_next"] = bull_vel
    out["vel_bull_fwd_8"]          = bull_vel
    out["Price_Velocity_Bear_next"] = bear_vel
    out["vel_bear_fwd_8"]          = bear_vel
    out["Price_Velocity_Net_next"]  = net_vel
    out["vel_net_fwd_8"]            = net_vel

    # ── Zone targets from SNR columns already in the CSV ───────────────────
    # Labels MAY use future bars (supervised targets). Features stay causal.
    def _col(name, default=np.nan):
        if name in df.columns:
            a = df[name].values.astype(np.float64)
            return a
        return np.full(n, default, dtype=np.float64)

    # Robust ATR: prefer column, else high-low rolling, else median |diff|
    atr_z = atr.copy()
    if not np.isfinite(atr_z).any() or np.nanstd(atr_z) < 1e-12:
        if high_col in df.columns and low_col in df.columns:
            atr_z = pd.Series(df[high_col].values - df[low_col].values).rolling(14, min_periods=1).mean().values
        else:
            atr_z = np.full(n, float(np.nanmedian(np.abs(np.diff(close, prepend=close[0])))) or 0.5)
    atr_z = np.nan_to_num(atr_z, nan=float(np.nanmedian(atr_z) if np.isfinite(atr_z).any() else 0.5))
    atr_z = np.maximum(atr_z, 1e-4)

    dist_s = np.nan_to_num(_col("snr_dist_support_5m"), nan=10.0, posinf=10.0)
    dist_r = np.nan_to_num(_col("snr_dist_resistance_5m"), nan=10.0, posinf=10.0)
    lvl_s = _col("snr_support_5m")
    lvl_r = _col("snr_resistance_5m")
    # reconstruct anchors if level prices missing
    if not np.isfinite(lvl_s).any():
        lvl_s = close - dist_s * atr_z
    if not np.isfinite(lvl_r).any():
        lvl_r = close + dist_r * atr_z
    lvl_s = np.nan_to_num(lvl_s, nan=np.nanmean(close))
    lvl_r = np.nan_to_num(lvl_r, nan=np.nanmean(close))

    LOOK = 12
    NEAR = 1.5  # ATR units — soft "interaction" threshold
    min_dist_fwd = np.full(n, 10.0, dtype=np.float64)
    bars_to_zone = np.full(n, float(LOOK), dtype=np.float64)
    zone_type = np.zeros(n, dtype=np.float64)

    # Vectorised-ish loop (n is moderate for training)
    for i in range(n):
        end = min(n, i + LOOK + 1)
        if end <= i + 1:
            continue
        a = atr_z[i] + 1e-8
        d_s = np.abs(close[i+1:end] - lvl_s[i]) / a
        d_r = np.abs(close[i+1:end] - lvl_r[i]) / a
        # also fold in *current* distance so target is never pure-10 when already near
        cur_s = float(dist_s[i]) if np.isfinite(dist_s[i]) else float(d_s[0]) if d_s.size else 10.0
        cur_r = float(dist_r[i]) if np.isfinite(dist_r[i]) else float(d_r[0]) if d_r.size else 10.0
        min_s = min(float(d_s.min()) if d_s.size else 10.0, cur_s)
        min_r = min(float(d_r.min()) if d_r.size else 10.0, cur_r)
        if min_s <= min_r:
            min_dist_fwd[i] = min_s
            zone_type[i] = 0.0
            hit = np.where(d_s <= NEAR)[0]
            bars_to_zone[i] = float(hit[0] + 1) if hit.size else (1.0 if cur_s <= NEAR else float(LOOK))
        else:
            min_dist_fwd[i] = min_r
            zone_type[i] = 1.0
            hit = np.where(d_r <= NEAR)[0]
            bars_to_zone[i] = float(hit[0] + 1) if hit.size else (1.0 if cur_r <= NEAR else float(LOOK))

    mid = 0.5 * (lvl_s + lvl_r)
    lo, hi = np.percentile(mid, 5), np.percentile(mid, 95)
    zone_idx = np.clip(((mid - lo) / (hi - lo + 1e-8) * 15.0), 0, 15).astype(np.float64)

    out["adv_target_next_zone_distance"] = np.clip(min_dist_fwd, 0.0, 10.0).astype(np.float32)
    out["adv_target_next_zone_bars"]     = np.clip(bars_to_zone, 1.0, float(LOOK)).astype(np.float32)
    out["adv_target_next_zone_type"]     = zone_type.astype(np.float32)
    out["adv_target_next_zone_idx"]      = zone_idx.astype(np.float32)

    return {k: np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0) for k, v in out.items()}


def _fill_ml_targets(df, n=None):
    n = n or len(df)
    synthesized = _synthesize_ml_targets(df)
    out = {}
    for k in _ML_KEYS:
        if k in df.columns:
            arr = np.nan_to_num(df[k].values.astype(np.float32), nan=0.0)
        else:
            arr = synthesized.get(k, np.zeros(n, dtype=np.float32))
        if len(arr) < n:
            arr = np.concatenate([arr, np.zeros(n - len(arr), dtype=np.float32)])
        out[k] = arr[:n]
    return out


train_ml_targets = _fill_ml_targets(train_df)
val_ml_targets = _fill_ml_targets(val_df) if val_df is not None else {k: np.zeros(1, np.float32) for k in _ML_KEYS}

non_zero = sum(1 for k, v in train_ml_targets.items() if float(np.abs(v).mean()) > 1e-6)
print(f"ML targets synthesized | non-zero keys: {non_zero} / {len(train_ml_targets)}")
for k, v in train_ml_targets.items():
    if float(np.abs(v).mean()) > 1e-6:
        print(f"  {k:45s}  mean={v.mean():.4f}  std={v.std():.4f}")


In [ ]:
# =============================================================================
# PHASE 1: META-LEARNER MULTI-HEAD TRAINING WITH 21+ ML TARGETS
#
# Updates:
#   1. Context windows: Meta=150 bars (12h of 5m data)
#   2. 6 primary heads + 21+ ML targets (zone, volatility, velocity)
#   3. Graceful fallback if ML targets not in CSV (zeros as defaults)
#   4. Multi-task loss with proper weighting
# =============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Target column preparation (synthesize if not in CSV)
for df in (train_df, val_df, test_df):
    if df is not None:
        df["forward_move_1"]  = df[close_col].shift(-1)  - df[close_col]
        df["forward_move_3"]  = df[close_col].shift(-3)  - df[close_col]
        df["forward_move_6"]  = df[close_col].shift(-6)  - df[close_col]
        df["forward_move_12"] = df[close_col].shift(-12) - df[close_col]
        df["target_dir_5m"]   = (df["forward_move_1"]  > 0).astype(np.float32)
        df["target_dir_15m"]  = (df["forward_move_3"]  > 0).astype(np.float32)
        df["target_dir_30m"]  = (df["forward_move_6"]  > 0).astype(np.float32)
        df["target_dir_1h"]   = (df["forward_move_12"] > 0).astype(np.float32)

print("[Bug1-fix] forward_move_1 std:", train_df["forward_move_1"].std(), "mean:", train_df["forward_move_1"].mean())
net = SignalMetaNetwork(input_dim=input_dim, num_features=num_features, hidden_dim=256).to(device)
target_net = SignalMetaNetwork(input_dim=input_dim, num_features=num_features, hidden_dim=256).to(device)
target_net.load_state_dict(net.state_dict())

optimizer = optim.AdamW(net.parameters(), lr=5e-4, weight_decay=1e-4)




N_train = len(train_df) - lookback_bars - 12
N_val   = len(val_df) - lookback_bars - 12 if val_df is not None else 0
META_EPOCHS = 80
BATCH_SIZE = 64
steps_per_epoch = max(1, N_train // BATCH_SIZE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=META_EPOCHS * steps_per_epoch, eta_min=1e-5)

train_num_matrix = np.nan_to_num(train_df[feature_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
val_num_matrix   = np.nan_to_num(val_df[feature_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0) if val_df is not None else None

def _extract_targets(df):
    """
    Extract targets for multi-head training (6 primary heads + 21+ ML targets from data pipeline).
    
    Returns: (q, pips, risk, rev, strength, liq, ml_targets_dict)
    """
    close_vals = df[close_col].values.astype(np.float32)
    
    # Dynamic True ATR computation
    if atr_col and atr_col in df.columns:
        atr_vals_local = df[atr_col].values.astype(np.float32)
    else:
        h_col = next((c for c in [high_col, "high_5m", "High", "high"] if c in df.columns), None)
        l_col = next((c for c in [low_col, "low_5m", "Low", "low"] if c in df.columns), None)
        if h_col and l_col:
            highs = df[h_col].values.astype(np.float32)
            lows  = df[l_col].values.astype(np.float32)
            atr_vals_local = pd.Series(highs - lows).rolling(14, min_periods=1).mean().values.astype(np.float32)
        else:
            atr_vals_local = close_vals * 0.0008  # 8 pips realistic 5m bar ATR fallback
    atr_vals_local = np.maximum(atr_vals_local, 1e-4)

    fwd = {
        1:  df["forward_move_1"].values.astype(np.float32),
        3:  df["forward_move_3"].values.astype(np.float32),
        6:  df["forward_move_6"].values.astype(np.float32),
        12: df["forward_move_12"].values.astype(np.float32),
    }
    dirs = {
        1:  df["target_dir_5m"].values.astype(np.float32),
        3:  df["target_dir_15m"].values.astype(np.float32),
        6:  df["target_dir_30m"].values.astype(np.float32),
        12: df["target_dir_1h"].values.astype(np.float32),
    }

    # Primary 6 Heads
    strength_cols = []
    for h_bars, h_key in zip([1, 3, 6, 12], [1, 3, 6, 12]):
        move_atr_signed = fwd[h_key] / (atr_vals_local * np.sqrt(h_bars))
        horizon_strength = 0.5 + 0.5 * np.clip(move_atr_signed / 1.5, -1.0, 1.0)
        horizon_strength = np.clip(horizon_strength.astype(np.float32), 0.05, 0.95)
        strength_cols.append(horizon_strength)
    strength_targets = np.column_stack(strength_cols)

    pips = np.column_stack([
        fwd[1] / atr_vals_local, fwd[3] / (atr_vals_local * np.sqrt(3)),
        fwd[6] / (atr_vals_local * np.sqrt(6)), fwd[12] / (atr_vals_local * np.sqrt(12)),
    ]).astype(np.float32)
    pips = np.clip(pips, -10.0, 10.0)

    risk = np.column_stack([
        np.maximum(fwd[1],  0) / atr_vals_local, np.maximum(-fwd[1],  0) / atr_vals_local,
        np.maximum(fwd[3],  0) / (atr_vals_local * np.sqrt(3)),  np.maximum(-fwd[3],  0) / (atr_vals_local * np.sqrt(3)),
        np.maximum(fwd[6],  0) / (atr_vals_local * np.sqrt(6)),  np.maximum(-fwd[6],  0) / (atr_vals_local * np.sqrt(6)),
        np.maximum(fwd[12], 0) / (atr_vals_local * np.sqrt(12)), np.maximum(-fwd[12], 0) / (atr_vals_local * np.sqrt(12)),
    ]).astype(np.float32)
    risk = np.clip(risk, 0.0, 10.0)

    liq = np.column_stack([
        np.abs(fwd[1]) / atr_vals_local,
        np.abs(fwd[3]) / (atr_vals_local * np.sqrt(3)),
    ]).astype(np.float32)
    liq = np.clip(liq, 0.0, 10.0)

    q = np.column_stack([dirs[1], dirs[3], dirs[6], dirs[12]]).astype(np.float32)
    q = np.clip(q, 0.0, 1.0)

    rev = (dirs[1] != dirs[3]).astype(np.float32).reshape(-1, 1)

    # 21+ Optional ML Targets (graceful fallback if not in CSV)
    ml_targets_dict = {}
    
    zone_cols = ["adv_target_next_zone_idx", "adv_target_next_zone_bars", "adv_target_next_zone_distance", "adv_target_next_zone_volume"]
    for col in zone_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    vol_cols = ["Volatility_Regime_next", "vol_regime_fwd_8", "Volatility_Expansion_next", "vol_expansion_fwd_8", 
                "Volatility_Bull_next", "Volatility_Bear_next", "Regime_Speed_Bull_next", "Regime_Speed_Bear_next", 
                "speed_aligned_fwd_8", "speed_divergence_fwd_8"]
    for col in vol_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    vel_cols = ["Price_Velocity_Bull_next", "vel_bull_fwd_8", "Price_Velocity_Bear_next", "vel_bear_fwd_8", 
                "Price_Velocity_Net_next", "vel_net_fwd_8"]
    for col in vel_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    csm_cols = ["adv_target_CSM_hist_fast_next", "adv_target_CSM_hist_slow_next", 
                "adv_target_CSM_asset_fast_next", "adv_target_CSM_dxy_fast_next"]
    for col in csm_cols:
        ml_targets_dict[col] = df[col].values.astype(np.float32) if col in df.columns else np.zeros(len(df), dtype=np.float32)
    
    return (
        np.nan_to_num(q, nan=0.0),
        np.nan_to_num(pips, nan=0.0),
        np.nan_to_num(risk, nan=0.0),
        np.nan_to_num(rev, nan=0.0),
        np.nan_to_num(strength_targets, nan=0.5),
        np.nan_to_num(liq, nan=0.0),
        ml_targets_dict,  # Return 21+ optional targets
    )


# =============================================================================
# TARGET REBUILD (fixes zero-variance strength/pips that blocked learning)
# =============================================================================
def _rebuild_targets(df):
    """Guaranteed non-degenerate soft strengths + ATR-scaled pips."""
    close = df[close_col].values.astype(np.float64)
    if atr_col and atr_col in df.columns:
        atr = df[atr_col].values.astype(np.float64)
        if not np.isfinite(atr).any() or np.nanstd(atr) < 1e-12:
            atr = pd.Series(df[high_col].values - df[low_col].values).rolling(14, min_periods=1).mean().values
    else:
        atr = pd.Series(df[high_col].values - df[low_col].values).rolling(14, min_periods=1).mean().values
    atr = np.asarray(atr, dtype=np.float64)
    atr = np.where(np.isfinite(atr) & (atr > 0), atr, np.nan)
    med = np.nanmedian(atr) if np.isfinite(np.nanmedian(atr)) else float(np.nanmedian(np.abs(np.diff(close))) or 0.5)
    atr = np.nan_to_num(atr, nan=med)
    atr = np.maximum(atr, 1e-4)

    # Rebuild forward moves from close (ignore broken CSV columns)
    fwd = {}
    for hb in (1, 3, 6, 12):
        f = np.zeros(len(df), dtype=np.float64)
        f[:-hb] = close[hb:] - close[:-hb]
        fwd[hb] = f
        df[f"forward_move_{hb}"] = f
    for hb, name in [(1, "target_dir_5m"), (3, "target_dir_15m"), (6, "target_dir_30m"), (12, "target_dir_1h")]:
        df[name] = (fwd[hb] > 0).astype(np.float32)

    strength = np.zeros((len(df), 4), dtype=np.float32)
    pips = np.zeros((len(df), 4), dtype=np.float32)
    risk = np.zeros((len(df), 8), dtype=np.float32)
    for hi, hb in enumerate((1, 3, 6, 12)):
        norm = fwd[hb] / (atr * np.sqrt(hb))
        norm = np.nan_to_num(norm, nan=0.0, posinf=0.0, neginf=0.0)
        strength[:, hi] = np.clip(0.5 + 0.5 * np.clip(norm / 1.5, -1.0, 1.0), 0.05, 0.95).astype(np.float32)
        pips[:, hi] = np.clip(norm, -10, 10).astype(np.float32)
        risk[:, hi * 2] = np.clip(np.maximum(norm, 0), 0, 10).astype(np.float32)
        risk[:, hi * 2 + 1] = np.clip(np.maximum(-norm, 0), 0, 10).astype(np.float32)

    q = np.column_stack([
        df["target_dir_5m"].values, df["target_dir_15m"].values,
        df["target_dir_30m"].values, df["target_dir_1h"].values,
    ]).astype(np.float32)
    q = np.clip(q, 0.0, 1.0)
    rev = (q[:, 0] != q[:, 1]).astype(np.float32).reshape(-1, 1)
    liq = np.column_stack([np.abs(pips[:, 0]), np.abs(pips[:, 1])]).astype(np.float32)
    return q, pips, risk, rev, strength, liq

train_targets_q, train_targets_pips, train_targets_risk, train_targets_rev, train_targets_strength, train_targets_liq = _rebuild_targets(train_df)
print("[FIXED targets] strength mean/std:", train_targets_strength.mean(0), train_targets_strength.std(0))
print("[FIXED targets] pips abs-mean:", np.abs(train_targets_pips).mean(0))
print("[FIXED targets] q mean:", train_targets_q.mean(0))
assert train_targets_strength.std(0).min() > 0.01, "Strength still has zero variance — abort"
assert np.abs(train_targets_pips).mean() > 1e-6, "Pips still zero — abort"

# ── Bug 3 fix: ATR-threshold direction weights ────────────────────────────────
# Only bars where abs(forward_move_1) > DIRECTION_THRESHOLD * ATR carry a real
# directional signal. Exclude micro-moves so the Q-head trains on discriminative
# examples, not a 50/50 coin flip.
DIRECTION_THRESHOLD = 0.30   # tunable; 0.3 ATR excludes ~40-50% of smallest moves
# Use ATR column only if it contains finite values; otherwise fall back to H-L rolling mean
_atr_raw = train_df[atr_col].values.astype(np.float32) if atr_col and atr_col in train_df.columns else None
_use_atr_col = _atr_raw is not None and np.isfinite(_atr_raw).any()
_atr_for_weight = _atr_raw if _use_atr_col else (
    pd.Series(train_df[high_col].values - train_df[low_col].values)
    .rolling(14, min_periods=1).mean().values.astype(np.float32)
)
_atr_for_weight = np.maximum(np.nan_to_num(_atr_for_weight, nan=0.001), 1e-4)
_fwd_move_1_abs = np.abs(train_df["forward_move_1"].values.astype(np.float32))
direction_weight = (_fwd_move_1_abs > DIRECTION_THRESHOLD * _atr_for_weight).astype(np.float32)

# Validate discriminability after ATR-threshold filter
_filtered_mask = direction_weight.astype(bool)
_effective_pos_rate = train_df["target_dir_5m"].values[_filtered_mask].mean() if _filtered_mask.sum() > 0 else 0.5
print(f"[Bug3-fix] Effective direction label positive rate: {_effective_pos_rate:.4f} "
      f"(on {_filtered_mask.sum():.0f} / {len(direction_weight)} bars)")
_bias = abs(_effective_pos_rate - 0.5)
print(f"[Bug3-fix] |rate - 0.5| = {_bias:.4f}  "
      f"({'OK discriminative' if _bias > 0.015 else 'near-balanced — consider lowering DIRECTION_THRESHOLD'})")

if val_df is not None:
    val_targets_q, val_targets_pips, val_targets_risk, val_targets_rev, val_targets_strength, val_targets_liq = _rebuild_targets(val_df)
else:
    val_targets_q = val_targets_pips = val_targets_risk = val_targets_rev = val_targets_strength = val_targets_liq = None

# ml_targets: REUSE synthesis from Cell 9 (or re-synthesize). NEVER zero-overwrite.
# Previous bug: this cell redefined _fill_ml_targets without the synthesizer, so all
# Volatility_*/Price_Velocity_*/Regime_Speed_* targets became zeros → Zone/Vol/Vel
# losses stayed 0.0 for the entire meta training run.
_ML_KEYS = [
    "adv_target_next_zone_idx", "adv_target_next_zone_bars", "adv_target_next_zone_distance",
    "adv_target_next_zone_type",
    "Volatility_Bull_next", "Volatility_Bear_next", "Volatility_Regime_next", "Volatility_Expansion_next",
    "Regime_Speed_Bull_next", "Regime_Speed_Bear_next",
    "speed_aligned_fwd_8", "speed_divergence_fwd_8",
    "Price_Velocity_Bull_next", "vel_bull_fwd_8", "Price_Velocity_Bear_next", "vel_bear_fwd_8",
    "Price_Velocity_Net_next", "vel_net_fwd_8",
    "adv_target_CSM_hist_fast_next", "adv_target_CSM_hist_slow_next",
    "adv_target_CSM_asset_fast_next", "adv_target_CSM_dxy_fast_next",
]

def _fill_ml_targets_phase1(df, n=None):
    """Prefer CSV columns; fall back to Cell-9 synthesizer; never leave real keys zeroed."""
    n = n or len(df)
    # synthesizer from Cell 9 (must already be defined)
    if "_synthesize_ml_targets" in globals():
        synthesized = _synthesize_ml_targets(df)
    else:
        synthesized = {k: np.zeros(n, dtype=np.float32) for k in _ML_KEYS}
    out = {}
    for k in _ML_KEYS:
        if k in df.columns:
            arr = np.nan_to_num(df[k].values.astype(np.float32), nan=0.0)
            # if the CSV column is entirely zero/constant, still prefer synthesizer
            if float(np.abs(arr).mean()) < 1e-8 and k in synthesized:
                arr = synthesized[k]
        else:
            arr = synthesized.get(k, np.zeros(n, dtype=np.float32))
        arr = np.asarray(arr, dtype=np.float32)
        if len(arr) < n:
            arr = np.concatenate([arr, np.zeros(n - len(arr), dtype=np.float32)])
        out[k] = np.nan_to_num(arr[:n], nan=0.0, posinf=0.0, neginf=0.0)
    return out

# If Cell 9 already populated train_ml_targets with non-zero keys, keep & refresh only zeros
_pre_nonzero = 0
if "train_ml_targets" in globals() and isinstance(train_ml_targets, dict):
    _pre_nonzero = sum(1 for k, v in train_ml_targets.items() if float(np.abs(v).mean()) > 1e-6)

train_ml_targets = _fill_ml_targets_phase1(train_df)
val_ml_targets = _fill_ml_targets_phase1(val_df) if val_df is not None else {k: np.zeros(1, np.float32) for k in _ML_KEYS}

# Scale velocity targets into a stable SmoothL1 range (ATR-normalised already, clip extremes)
for _vk in ("Price_Velocity_Bull_next", "Price_Velocity_Bear_next", "Price_Velocity_Net_next",
            "vel_bull_fwd_8", "vel_bear_fwd_8", "vel_net_fwd_8",
            "Regime_Speed_Bull_next", "Regime_Speed_Bear_next"):
    if _vk in train_ml_targets:
        train_ml_targets[_vk] = np.clip(train_ml_targets[_vk], -10.0, 10.0).astype(np.float32)
    if _vk in val_ml_targets:
        val_ml_targets[_vk] = np.clip(val_ml_targets[_vk], -10.0, 10.0).astype(np.float32)

_nz = sum(1 for k, v in train_ml_targets.items() if float(np.abs(v).mean()) > 1e-6)
print(f"ML targets filled | non-zero keys: {_nz} / {len(train_ml_targets)}  (pre-existing non-zero from Cell 9: {_pre_nonzero})")
for k, v in train_ml_targets.items():
    if float(np.abs(v).mean()) > 1e-6:
        print(f"  {k:45s}  mean={v.mean():+.4f}  std={v.std():.4f}")
assert _nz >= 6, (
    "CRITICAL: ML vol/vel targets still all-zero after synthesis. "
    "Ensure Cell 9 (_synthesize_ml_targets) ran successfully."
)

# =============================================================================
# SIGNAL-AWARE META SUPERVISION
# Ties meta Q/strength to confluence (same signal family as Phase-2 shaping).
#   (a) sample weights — train harder on high-confluence directional bars
#   (b) strength targets reshaped so conviction tracks quality
# Pseudo-action = future direction label (up→CALL quality, down→PUT quality).
# Signal features are concurrent CSV columns only (no lookahead in features).
# =============================================================================
META_SIGNAL_WEIGHT = 0.6
META_STRENGTH_BLEND = 0.45
META_LIQ_COEF = 0.08              # was 0.2; liq was dominating total loss
META_EARLY_STOP_PATIENCE = 12
META_MIN_EPOCHS = 8

def precompute_meta_signal_quality(df, targets_q):
    """Per-bar, per-horizon quality ∈ [0,1]. targets_q shape (N, 4)."""
    n = len(df)
    out = np.zeros((n, 4), dtype=np.float32)

    def col(name, default=0.0):
        if name in df.columns:
            a = df[name].values.astype(np.float64)
            return np.nan_to_num(a, nan=default, posinf=default, neginf=default)
        return np.full(n, float(default), dtype=np.float64)

    rsi = col("rsi_diff_5m", np.nan)
    if not np.isfinite(rsi).any():
        rsi = col("mtf_rsi_diff", 0.0)
    rsi_n = np.clip(rsi / 200.0, -1.0, 1.0)
    near_s = np.clip(1.0 - col("snr_dist_support_5m", 10.0) / 4.0, 0.0, 1.0)
    near_r = np.clip(1.0 - col("snr_dist_resistance_5m", 10.0) / 4.0, 0.0, 1.0)
    mtf_c = np.clip(col("mtf_snr_confluence", 0.0) / 2.0, 0.0, 1.0)
    dxy_near_s = np.clip(1.0 - col("dxy_snr_dist_support_5m", 10.0) / 4.0, 0.0, 1.0)
    dxy_near_r = np.clip(1.0 - col("dxy_snr_dist_resistance_5m", 10.0) / 4.0, 0.0, 1.0)
    r_bull = col("Regime_Speed_Bull_5m", np.nan)
    r_bear = col("Regime_Speed_Bear_5m", np.nan)

    tq = np.asarray(targets_q, dtype=np.float64)
    n_h = min(4, tq.shape[1]) if tq.ndim == 2 else 0
    for h in range(n_h):
        is_call = tq[:, h] >= 0.5
        dir_q = np.where(is_call, 0.5 + 0.5 * rsi_n, 0.5 - 0.5 * rsi_n)
        zone_q = np.where(is_call, near_s, near_r)
        dxy_a = np.where(is_call, dxy_near_r, dxy_near_s)
        conf = 0.6 * mtf_c + 0.4 * dxy_a
        rb = np.where(is_call, r_bull, r_bear)
        has_r = np.isfinite(rb)
        conf = np.where(
            has_r,
            0.7 * conf + 0.3 * np.clip(np.nan_to_num(rb, nan=0.0), 0.0, 1.0),
            conf,
        )
        out[:, h] = np.clip(0.35 * dir_q + 0.30 * zone_q + 0.35 * conf, 0.0, 1.0).astype(np.float32)
    return out


print("[Meta signal] Precomputing train signal-quality scores...")
train_sig_q = precompute_meta_signal_quality(train_df, train_targets_q)
print(f"  train sig_q mean/std per h: mean={train_sig_q.mean(0)} std={train_sig_q.std(0)}")
print(f"  train sig_q overall mean={float(train_sig_q.mean()):.4f}")

train_targets_strength_raw = train_targets_strength.copy()
train_targets_strength = np.clip(
    0.5 + (train_targets_strength_raw - 0.5) * (0.5 + 0.5 * train_sig_q),
    0.05, 0.95,
).astype(np.float32)
print(f"[Meta signal] strength reshape: raw_std={train_targets_strength_raw.std(0)} "
      f"→ shaped_std={train_targets_strength.std(0)}")

train_sig_q_bar = train_sig_q.mean(axis=1).astype(np.float32)
signal_sample_weight = (
    (1.0 - META_SIGNAL_WEIGHT) * direction_weight
    + META_SIGNAL_WEIGHT * direction_weight * train_sig_q_bar
).astype(np.float32)
print(f"[Meta signal] sample_weight mean={float(signal_sample_weight.mean()):.4f} "
      f"(dir_weight mean={float(direction_weight.mean()):.4f})")

if val_df is not None and val_targets_q is not None:
    val_sig_q = precompute_meta_signal_quality(val_df, val_targets_q)
    val_targets_strength = np.clip(
        0.5 + (val_targets_strength - 0.5) * (0.5 + 0.5 * val_sig_q),
        0.05, 0.95,
    ).astype(np.float32)
    print(f"  val sig_q mean={float(val_sig_q.mean()):.4f}")
else:
    val_sig_q = None


best_val_avg_wr = -1.0  # maximize val WR (strict > only)
best_meta_weights = None
epochs_without_improve = 0

print(f"[Phase 1] Meta-Learner Training: up to {META_EPOCHS} epochs, {steps_per_epoch} steps/epoch")
print(f"  Early-stop patience={META_EARLY_STOP_PATIENCE} (min epochs={META_MIN_EPOCHS})")
print(f"  Signal-aware: META_SIGNAL_WEIGHT={META_SIGNAL_WEIGHT}, LIQ_COEF={META_LIQ_COEF}")
print(f"  {'Epoch':>6} | {'AvgLoss':>9} | {'Q':>8} | {'Str':>8} | {'Pips':>8} | {'Risk':>8} | {'Liq':>8} | {'Rev':>8} | {'Sel':>8} | {'5mWR':>6} {'15mWR':>6} {'30mWR':>6} {'1hWR':>6} AvgWR | Status")
print(f"  {'-'*165}")

for ep in range(META_EPOCHS):
    indices = list(range(N_train))
    random.shuffle(indices)

    ep_tot, ep_q, ep_str, ep_pips, ep_risk, ep_liq, ep_rev, ep_sel = 0., 0., 0., 0., 0., 0., 0., 0.
    ep_zone, ep_vol, ep_vel = 0., 0., 0.
    epoch_steps = 0

    net.train()
    for b_start in range(0, N_train, BATCH_SIZE):
        batch_idx = indices[b_start: b_start + BATCH_SIZE]
        if len(batch_idx) < BATCH_SIZE:
            continue

        ti = np.array(batch_idx) + lookback_bars
        x_batch = np.stack([train_num_matrix[i: i + lookback_bars].flatten() for i in batch_idx])

        x_t      = torch.tensor(x_batch, dtype=torch.float32, device=device)
        y_q_t    = torch.tensor(train_targets_q[ti],        dtype=torch.float32, device=device)
        y_pips_t = torch.tensor(train_targets_pips[ti],     dtype=torch.float32, device=device)
        y_risk_t = torch.tensor(train_targets_risk[ti],     dtype=torch.float32, device=device)
        y_rev_t  = torch.tensor(train_targets_rev[ti],      dtype=torch.float32, device=device)
        y_str_t  = torch.tensor(train_targets_strength[ti], dtype=torch.float32, device=device)
        y_liq_t  = torch.tensor(train_targets_liq[ti],      dtype=torch.float32, device=device)
        
        # Load 21+ ML targets
        y_zone_idx = torch.tensor(train_ml_targets["adv_target_next_zone_idx"][ti], dtype=torch.float32, device=device)
        y_zone_bars = torch.tensor(train_ml_targets["adv_target_next_zone_bars"][ti], dtype=torch.float32, device=device)
        y_zone_dist = torch.tensor(train_ml_targets["adv_target_next_zone_distance"][ti], dtype=torch.float32, device=device)
        y_zone_type = torch.tensor(train_ml_targets.get("adv_target_next_zone_type", np.zeros(len(train_df)))[ti], dtype=torch.float32, device=device)
        y_vol_regime = torch.tensor(train_ml_targets["Volatility_Regime_next"][ti], dtype=torch.float32, device=device)
        y_vol_exp = torch.tensor(train_ml_targets["Volatility_Expansion_next"][ti], dtype=torch.float32, device=device)
        y_vol_bull = torch.tensor(train_ml_targets["Volatility_Bull_next"][ti], dtype=torch.float32, device=device)
        y_vol_bear = torch.tensor(train_ml_targets["Volatility_Bear_next"][ti], dtype=torch.float32, device=device)
        y_vel_bull = torch.tensor(train_ml_targets["Price_Velocity_Bull_next"][ti], dtype=torch.float32, device=device)
        y_vel_bear = torch.tensor(train_ml_targets["Price_Velocity_Bear_next"][ti], dtype=torch.float32, device=device)
        y_vel_net = torch.tensor(train_ml_targets["Price_Velocity_Net_next"][ti], dtype=torch.float32, device=device)

        optimizer.zero_grad()
        q_vals, strength, pips, risk, liq, rev, aux1, aux2, selector_logits = net(x_t, return_aux=True)

        # Primary 6-head losses — signal-aware sample weights
        _dw_batch = torch.tensor(signal_sample_weight[ti], dtype=torch.float32, device=device)
        _dw_exp = _dw_batch.unsqueeze(1).expand_as(q_vals)
        _dw_sum = _dw_exp.sum() + 1e-6
        l_q = (_dw_exp * (q_vals - y_q_t) ** 2).sum() / _dw_sum
        _sw = _dw_batch.unsqueeze(1).expand_as(strength)
        l_str = ((_sw * (strength - y_str_t) ** 2).sum() / (_sw.sum() + 1e-6))
        l_pips = nn.SmoothL1Loss()(pips, y_pips_t)
        l_risk = nn.SmoothL1Loss()(risk[..., :y_risk_t.shape[-1]], y_risk_t) if risk.shape[-1] >= y_risk_t.shape[-1] else nn.SmoothL1Loss()(risk, y_risk_t[..., :risk.shape[-1]])
        l_liq  = nn.MSELoss()(liq, y_liq_t)
        l_rev  = nn.MSELoss()(rev, y_rev_t)

        true_best_h = y_str_t.argmax(dim=1).long()
        l_sel_base = nn.CrossEntropyLoss()(selector_logits, true_best_h)
        selector_probs = torch.softmax(selector_logits, dim=-1)
        entropy_sel = -torch.sum(selector_probs * torch.log(selector_probs + 1e-8), dim=-1).mean()
        l_sel = l_sel_base - 0.10 * entropy_sel

        target_aux = torch.cat([y_q_t, y_rev_t], dim=1)
        l_aux1 = nn.SmoothL1Loss()(aux1, target_aux) if aux1.shape[-1] == target_aux.shape[-1] else torch.tensor(0.0, device=device)
        l_aux2 = nn.SmoothL1Loss()(aux2, target_aux) if aux2.shape[-1] == target_aux.shape[-1] else torch.tensor(0.0, device=device)
        
        # 21+ ML target losses — GATED: skip any loss whose target is all-zero
        # Zone targets are now synthesized from SNR columns (distance/bars/type/idx).
        _y_zone_idx_mean = float(y_zone_idx.abs().mean())
        _y_zone_bars_mean = float(y_zone_bars.abs().mean())
        _y_zone_dist_mean = float(y_zone_dist.abs().mean())
        _y_zone_type_mean = float(y_zone_type.abs().mean()) if y_zone_type is not None else 0.0
        l_zone = torch.tensor(0.0, device=device)
        # distance → liquidity head (how near the next zone interaction is)
        if _y_zone_dist_mean > 1e-7 and liq.shape[-1] >= 1:
            # normalise dist 0–10 → roughly 0–1 for stable scale vs liq
            l_zone = l_zone + nn.SmoothL1Loss()(
                liq[:, 0:1], (y_zone_dist / 10.0).unsqueeze(1)
            ) * 0.08
        # bars-to-zone → pips head slot 0 (time-to-event proxy)
        if _y_zone_bars_mean > 1e-7 and pips.shape[-1] >= 1:
            l_zone = l_zone + nn.SmoothL1Loss()(
                pips[:, 0:1], (y_zone_bars / 12.0).unsqueeze(1)
            ) * 0.05
        # zone type (0=support, 1=resistance) → strength head slot as soft class
        if _y_zone_type_mean > 1e-7 and strength.shape[-1] >= 2:
            l_zone = l_zone + nn.MSELoss()(
                strength[:, 2:3], y_zone_type.unsqueeze(1)
            ) * 0.05
        # zone idx (0–15 bucket) → q_vals mean as weak regulariser
        if _y_zone_idx_mean > 1e-7:
            l_zone = l_zone + nn.MSELoss()(
                q_vals.mean(dim=1, keepdim=True), (y_zone_idx / 15.0).unsqueeze(1)
            ) * 0.03

        _y_vol_regime_mean = float(y_vol_regime.abs().mean())
        _y_vol_exp_mean = float(y_vol_exp.abs().mean())
        _y_vol_bull_mean = float(y_vol_bull.abs().mean())
        _y_vol_bear_mean = float(y_vol_bear.abs().mean())
        l_vol = torch.tensor(0.0, device=device)
        if _y_vol_regime_mean > 1e-7:
            l_vol = l_vol + nn.MSELoss()(strength[:, 0:1], y_vol_regime.unsqueeze(1)) * 0.05
        if _y_vol_exp_mean > 1e-7:
            # Expansion may be near-constant (~1.0); only use if it has real variance
            if float(y_vol_exp.std()) > 1e-4:
                l_vol = l_vol + nn.MSELoss()(strength[:, 1:2], y_vol_exp.unsqueeze(1)) * 0.05
        if _y_vol_bull_mean > 1e-7 and pips.shape[-1] >= 1:
            l_vol = l_vol + nn.SmoothL1Loss()(pips[:, 0:1], y_vol_bull.unsqueeze(1)) * 0.03
        if _y_vol_bear_mean > 1e-7 and pips.shape[-1] >= 2:
            l_vol = l_vol + nn.SmoothL1Loss()(pips[:, 1:2], y_vol_bear.unsqueeze(1)) * 0.03

        _y_vel_bull_mean = float(y_vel_bull.abs().mean())
        _y_vel_bear_mean = float(y_vel_bear.abs().mean())
        _y_vel_net_mean  = float(y_vel_net.abs().mean())
        l_vel = torch.tensor(0.0, device=device)
        if _y_vel_bull_mean > 1e-7:
            l_vel = l_vel + nn.SmoothL1Loss()(pips[:, 1:2], y_vel_bull.unsqueeze(1)) * 0.05
        if _y_vel_bear_mean > 1e-7:
            l_vel = l_vel + nn.SmoothL1Loss()(pips[:, 2:3], y_vel_bear.unsqueeze(1)) * 0.05
        if _y_vel_net_mean > 1e-7:
            l_vel = l_vel + nn.SmoothL1Loss()(pips[:, 3:4], y_vel_net.unsqueeze(1)) * 0.05

        loss = (
            l_q + 1.0 * l_str +
            0.3 * l_pips + 0.3 * l_risk + META_LIQ_COEF * l_liq +
            0.3 * l_rev +
            0.1 * l_aux1 + 0.1 * l_aux2 +
            0.5 * l_sel +
            0.20 * l_zone +
            0.15 * l_vol +
            0.15 * l_vel
        )

        if torch.isnan(loss):
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            for tp, p in zip(target_net.parameters(), net.parameters()):
                tp.data.copy_(0.005 * p.data + 0.995 * tp.data)

        ep_tot  += loss.item()
        ep_q    += l_q.item()
        ep_str  += l_str.item()
        ep_pips += l_pips.item()
        ep_risk += l_risk.item()
        ep_liq  += l_liq.item()
        ep_rev  += l_rev.item()
        ep_sel  += l_sel.item()
        ep_zone += l_zone.item()
        ep_vol  += l_vol.item()
        ep_vel  += l_vel.item()
        epoch_steps += 1

        if epoch_steps % 100 == 0 or epoch_steps == steps_per_epoch:
            print(f"  [Ep {ep+1:>2}/{META_EPOCHS} | Step {epoch_steps:>4}/{steps_per_epoch}] Tot={loss.item():.4e} Q={l_q.item():.4e} Str={l_str.item():.4e} Zone={l_zone.item():.4e} Vol={l_vol.item():.4e} Vel={l_vel.item():.4e}")

    s = max(epoch_steps, 1)
    avg_tot  = ep_tot / s
    avg_q    = ep_q / s
    avg_str  = ep_str / s
    avg_pips = ep_pips / s
    avg_risk = ep_risk / s
    avg_liq  = ep_liq / s
    avg_rev  = ep_rev / s
    avg_sel  = ep_sel / s
    avg_zone = ep_zone / s
    avg_vol  = ep_vol / s
    avg_vel  = ep_vel / s

    val_corrects = [0, 0, 0, 0]
    val_count = 0
    val_str_dist = np.zeros(4)

    if N_val > 0 and val_num_matrix is not None and val_targets_strength is not None:
        net.eval()
        with torch.no_grad():
            for v_start in range(0, N_val, BATCH_SIZE):
                v_end = min(v_start + BATCH_SIZE, N_val)
                v_idx = list(range(v_start, v_end))
                if not v_idx:
                    continue
                vti = np.array(v_idx) + lookback_bars
                vx_b = np.stack([val_num_matrix[i: i + lookback_bars].flatten() for i in v_idx])
                vx_t = torch.tensor(vx_b, dtype=torch.float32, device=device)
                vy_q_t   = torch.tensor(val_targets_q[vti],        dtype=torch.float32, device=device)
                vy_str_t = torch.tensor(val_targets_strength[vti], dtype=torch.float32, device=device)
                
                vq_vals, vstr, _, _, _, _, _, _, _ = net(vx_t, return_aux=True)
                
                for h in range(4):
                    pred_dir = (vq_vals[:, h] > 0.5).long()
                    true_dir = (vy_q_t[:, h] > 0.5).long()
                    val_corrects[h] += (pred_dir == true_dir).sum().item()
                val_count += len(v_idx)

        if val_count > 0:
            val_wrs = [c / val_count for c in val_corrects]
            val_avg_wr = float(np.mean(val_wrs))
            improved = val_avg_wr > best_val_avg_wr + 1e-12  # strict improvement only
            if improved:
                best_val_avg_wr = val_avg_wr
                best_meta_weights = {k: v.cpu().clone() for k, v in net.state_dict().items()}
                epochs_without_improve = 0
                status = "✓ NEW BEST"
            else:
                epochs_without_improve += 1
                status = f"(best={best_val_avg_wr:.2%} p={epochs_without_improve})"
        else:
            val_wrs = [0, 0, 0, 0]
            val_avg_wr = 0.0
            status = ""
            epochs_without_improve += 1
    else:
        val_wrs = [0, 0, 0, 0]
        val_avg_wr = 0.0
        status = ""
        epochs_without_improve += 1

    print(f"  {ep+1:>6} | {avg_tot:>9.4e} | {avg_q:>8.4e} | {avg_str:>8.4e} | {avg_pips:>8.4e} | {avg_risk:>8.4e} | {avg_liq:>8.4e} | {avg_rev:>8.4e} | {avg_sel:>8.4e} | {val_wrs[0]:>6.2%} {val_wrs[1]:>6.2%} {val_wrs[2]:>6.2%} {val_wrs[3]:>6.2%} {val_avg_wr:>6.2%} {status}")

    # Early stopping: only after META_MIN_EPOCHS, and only if we have a real best
    if (
        ep + 1 >= META_MIN_EPOCHS
        and epochs_without_improve >= META_EARLY_STOP_PATIENCE
        and best_meta_weights is not None
    ):
        print(f"\n⏹ Early stop at epoch {ep+1}: no val-WR improvement for "
              f"{epochs_without_improve} epochs (best={best_val_avg_wr:.2%})")
        break

if best_meta_weights:
    net.load_state_dict(best_meta_weights)
    print(f"\n✓ Loaded best meta-learner weights (Avg WR={best_val_avg_wr:.2%})")
else:
    print("\n⚠ No validation improvement recorded — keeping final epoch weights")

print(f"\n✓ Meta-Learner Training Complete")
print(f"  Final training loss: {avg_tot:.4e}")
print(f"  Best validation Avg WR: {best_val_avg_wr:.2%}")



In [ ]:
BUFFER_CAPACITY = 3000
BATCH_SIZE_Q = 64
Q_EPOCHS = 20
# =============================================================================
# PHASE 2: PER-HORIZON Q-LEARNING
# Architecture: 4 independent Q-heads (one per horizon), each with 3 actions
#               WAIT(0) / CALL(1) / PUT(2). Auto-expiry handles settlement.
# Gate: max 1 open position per horizon simultaneously.
# Training: each horizon's head is supervised only by that horizon's reward signal.
# =============================================================================
NUM_HORIZONS = NUM_HORIZONS if 'NUM_HORIZONS' in dir() else 4

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.parallel import DataParallel
import time
import numpy as np
import random
import pandas as pd

assert 'Q_LOOKBACK' in dir(), "Q_LOOKBACK not defined — run the config cell (Cell 2) first"
print(f"[Config check] Using Q_LOOKBACK = {Q_LOOKBACK}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

HORIZON_BARS_LIST = [1, 3, 6, 12]       # 5m, 15m, 30m, 1h
HORIZON_LABELS    = ["5m", "15m", "30m", "1h"]
NUM_HORIZONS      = 4
H_WAIT, H_CALL, H_PUT = 0, 1, 2         # Per-head action indices

# =============================================================================
# SIGNAL-QUALITY SCORING + REWARD SHAPING  (bridges Q-learner to confluence)
# Uses columns that ACTUALLY exist in the Kaggle CSVs / backend pipeline:
#   mtf_snr_confluence, rsi_diff_*, snr_dist_* / dxy_snr_dist_*, Regime_Speed_*,
#   and optionally cross_*/regime_* if evaluate_option_expiries features are present.
# Math is scaled for continuous fwd_pct rewards (not binary ±1).
# =============================================================================
ALIGNMENT_WEIGHT = 0.35          # how strongly quality biases the reward
SHAPING_SCALE    = 0.005         # absolute shaping magnitude (matches cost scale)

def _safe_get(row, key, default=0.0):
    """Pull a column if present; otherwise return default. Works with Series or dict.
    If default is None and the key is missing / NaN, returns None (caller must handle)."""
    try:
        if hasattr(row, "get"):
            v = row.get(key, default)
        else:
            v = row[key] if key in getattr(row, "index", []) else default
        if v is None:
            return default
        fv = float(v)
        if not np.isfinite(fv):
            return default
        return fv
    except Exception:
        return default


def _norm_signed(x, scale=50.0):
    """Map a possibly huge signed feature into [-1, 1]."""
    return float(np.clip(float(x) / (scale + 1e-8), -1.0, 1.0))


def compute_signal_quality_score(
    action,
    meta_strength_h,
    dir_flag,
    supp_dist,
    res_dist,
    vol_delta_ratio,
    row=None,
):
    """
    Quality ∈ [0, 1] — how well the chosen action aligns with available signals.

    Primary components (always available from static state + CSV):
      1. Meta conviction for this horizon                     25%
      2. Directional agreement (dir_flag / RSI-diff vs action) 25%
      3. Zone proximity appropriateness                        20%
      4. Volume confirmation                                   10%
      5. MTF SNR confluence + DXY-zone alignment               20%

    Extra boosts when evaluate_option_expiries-style columns exist
    (cross_index_signal, regime_strong_*, zone_bounce_signal, …).
    """
    # --- 1. Meta conviction -------------------------------------------------
    meta_q = float(np.clip((float(meta_strength_h) - 0.45) / 0.25, 0.0, 1.0))

    # --- 2. Direction alignment (dir_flag + RSI-diff if present) ------------
    if action == 1:  # CALL
        dir_q = 0.5 + 0.5 * float(np.clip(dir_flag, -1.0, 1.0))
    elif action == 2:  # PUT
        dir_q = 0.5 - 0.5 * float(np.clip(dir_flag, -1.0, 1.0))
    else:  # WAIT
        dir_q = 1.0 - abs(float(dir_flag)) * 0.5

    if row is not None:
        # rsi_diff_5m / mtf_rsi_diff can be huge in the CSV — normalise hard
        rsi_d = _safe_get(row, "rsi_diff_5m", None)
        if rsi_d is None:
            rsi_d = _safe_get(row, "mtf_rsi_diff", 0.0)
        rsi_n = _norm_signed(rsi_d, scale=200.0)  # positive → asset stronger than DXY
        if action == 1:
            dir_q = 0.6 * dir_q + 0.4 * (0.5 + 0.5 * rsi_n)
        elif action == 2:
            dir_q = 0.6 * dir_q + 0.4 * (0.5 - 0.5 * rsi_n)
        # WAIT stays as-is (mixed RSI is fine for waiting)

    # --- 3. Zone proximity (strategy is zone-anchored) ----------------------
    prox_band = 0.004
    near_supp = 1.0 if supp_dist <= prox_band else max(0.0, 1.0 - (supp_dist - prox_band) / 0.01)
    near_res  = 1.0 if res_dist  <= prox_band else max(0.0, 1.0 - (res_dist  - prox_band) / 0.01)
    # Prefer the *precomputed* SNR distances from the CSV when available
    # (they are ATR-normalised 0–10; closer to 0 = nearer the zone)
    if row is not None:
        s_csv = _safe_get(row, "snr_dist_support_5m", None)
        r_csv = _safe_get(row, "snr_dist_resistance_5m", None)
        if s_csv is not None:
            near_supp = float(np.clip(1.0 - s_csv / 4.0, 0.0, 1.0))
        if r_csv is not None:
            near_res = float(np.clip(1.0 - r_csv / 4.0, 0.0, 1.0))

    if action == 1:
        zone_q = near_supp
    elif action == 2:
        zone_q = near_res
    else:
        zone_q = 1.0 - 0.5 * max(near_supp, near_res)

    # --- 4. Volume confirmation ---------------------------------------------
    vd = float(np.clip(vol_delta_ratio, -1.0, 1.0))
    if action == 1:
        vol_q = 0.5 + 0.5 * vd
    elif action == 2:
        vol_q = 0.5 - 0.5 * vd
    else:
        vol_q = 0.5

    # --- 5. MTF SNR confluence + DXY zone alignment -------------------------
    conf_q = 0.5  # neutral default
    if row is not None:
        mtf_c = _safe_get(row, "mtf_snr_confluence", None)
        if mtf_c is not None:
            # CSV values observed ~0–3; map 0→0, ≥2→1
            conf_q = float(np.clip(mtf_c / 2.0, 0.0, 1.0))

        # DXY SNR proximity: if DXY is also at a matching side zone, boost
        dxy_s = _safe_get(row, "dxy_snr_dist_support_5m", 10.0)
        dxy_r = _safe_get(row, "dxy_snr_dist_resistance_5m", 10.0)
        if action == 1:
            # CALL benefits when DXY is weak / near its resistance (DXY down → asset up)
            dxy_align = float(np.clip(1.0 - dxy_r / 4.0, 0.0, 1.0))
        elif action == 2:
            dxy_align = float(np.clip(1.0 - dxy_s / 4.0, 0.0, 1.0))
        else:
            dxy_align = 0.5
        conf_q = 0.6 * conf_q + 0.4 * dxy_align

        # Regime speed (if non-NaN in full data)
        r_bull = _safe_get(row, "Regime_Speed_Bull_5m", None)
        r_bear = _safe_get(row, "Regime_Speed_Bear_5m", None)
        if r_bull is not None and np.isfinite(r_bull):
            if action == 1:
                conf_q = 0.7 * conf_q + 0.3 * float(np.clip(r_bull, 0, 1))
            elif action == 2 and r_bear is not None and np.isfinite(r_bear):
                conf_q = 0.7 * conf_q + 0.3 * float(np.clip(r_bear, 0, 1))

    quality = (
        0.25 * meta_q +
        0.25 * dir_q +
        0.20 * zone_q +
        0.10 * vol_q +
        0.20 * conf_q
    )

    # --- Optional evaluate_option_expiries boosts ---------------------------
    if row is not None:
        cross_sym = _safe_get(row, "cross_index_signal", None)
        cross_dxy = _safe_get(row, "cross_dxy_signal", None)
        if cross_sym is not None and cross_dxy is not None:
            if action == 1:
                cross_agree = 0.5 * (1.0 + np.clip(cross_sym, -1, 1)) + 0.5 * (1.0 + np.clip(cross_dxy, -1, 1))
                cross_agree /= 2.0
            elif action == 2:
                cross_agree = 0.5 * (1.0 - np.clip(cross_sym, -1, 1)) + 0.5 * (1.0 - np.clip(cross_dxy, -1, 1))
                cross_agree /= 2.0
            else:
                cross_agree = 0.5
            quality = 0.80 * quality + 0.20 * float(cross_agree)

        r_strong_asset = _safe_get(row, "regime_strong_asset", 0.0)
        r_strong_dxy   = _safe_get(row, "regime_strong_dxy", 0.0)
        if action == 1 and r_strong_asset > 0.5:
            quality = min(1.0, quality + 0.08)
        if action == 2 and r_strong_dxy > 0.5:
            quality = min(1.0, quality + 0.08)

        if action == 1:
            zb = _safe_get(row, "zone_bounce_signal", None)
            if zb is not None:
                quality = 0.85 * quality + 0.15 * float(np.clip(zb, 0, 1))
        elif action == 2:
            zr = _safe_get(row, "zone_rejection_signal", None)
            if zr is not None:
                quality = 0.85 * quality + 0.15 * float(np.clip(zr, 0, 1))

    return float(np.clip(quality, 0.0, 1.0))


def compute_shaped_reward(base_reward, signal_quality, alignment_weight=ALIGNMENT_WEIGHT, scale=SHAPING_SCALE):
    """
    Continuous-reward shaping that preserves the original magnitude order.

    quality_centered ∈ [-1, +1]
      +1 → perfect alignment  → push reward upward
      -1 → total misalignment → push reward downward

    Final magnitude stays inside the same clip band the rest of the loop uses.
    """
    q_c = 2.0 * float(signal_quality) - 1.0          # [-1, +1]
    shaped = float(base_reward) + alignment_weight * scale * q_c
    return float(np.clip(shaped, -0.06, 0.06))

print("[Signal shaping] compute_signal_quality_score + compute_shaped_reward ready "
      f"(ALIGNMENT_WEIGHT={ALIGNMENT_WEIGHT}, SHAPING_SCALE={SHAPING_SCALE})")


net.eval()
if n_gpus > 1:
    meta_dp = DataParallel(net)
else:
    meta_dp = net

# ---------- Precompute meta outputs ----------
PRECOMPUTE_BATCH = 256
N_train = len(train_df) - lookback_bars - 12
print(f"[Precompute] Meta features for {N_train} steps...")
t0 = time.time()

meta_strengths = np.zeros((N_train, 4), dtype=np.float32)
meta_qmax      = np.zeros(N_train, dtype=np.float32)
meta_rev       = np.zeros(N_train, dtype=np.float32)
meta_mfe       = np.zeros(N_train, dtype=np.float32)
meta_mae       = np.zeros(N_train, dtype=np.float32)

with torch.no_grad():
    for start in range(0, N_train, PRECOMPUTE_BATCH):
        end = min(start + PRECOMPUTE_BATCH, N_train)
        batch_x = np.stack([train_num_matrix[i: i + lookback_bars].flatten() for i in range(start, end)])
        x_t = torch.tensor(batch_x, dtype=torch.float32, device=device)
        q_vals, strength, pips, risk, liq, rev = meta_dp(x_t)
        meta_strengths[start:end] = strength.cpu().numpy()
        meta_qmax[start:end]      = q_vals.max(dim=1).values.cpu().numpy()
        meta_rev[start:end]       = rev.squeeze(-1).cpu().numpy() if rev.ndim > 1 else rev.cpu().numpy()
        if risk.shape[-1] >= 2:
            meta_mfe[start:end]   = risk[:, 0].cpu().numpy()
            meta_mae[start:end]   = risk[:, 1].cpu().numpy()
        if (start // PRECOMPUTE_BATCH) % 20 == 0:
            print(f"  meta precompute {end}/{N_train}")

meta_strengths = np.nan_to_num(meta_strengths, nan=0.5, posinf=1.0, neginf=0.0)
meta_qmax      = np.nan_to_num(meta_qmax, nan=0.5)
meta_rev       = np.nan_to_num(meta_rev, nan=0.2)
meta_mfe       = np.nan_to_num(meta_mfe, nan=0.5)
meta_mae       = np.nan_to_num(meta_mae, nan=0.15)
print(f"Meta precompute done in {time.time()-t0:.1f}s")

# ---------- Precompute zones ----------
print("[Precompute] SNR zones...")
t1 = time.time()
price_data_hl = train_df[[open_col, high_col, low_col, close_col, vol_col]].rename(
    columns={open_col: "Open", high_col: "High", low_col: "Low", close_col: "Close", vol_col: "Volume"}
)
nearest_supp_list = [None] * N_train
nearest_res_list  = [None] * N_train
close_prices = train_df[close_col].values.astype(np.float64)
if atr_col and atr_col in train_df.columns:
    atr_vals = train_df[atr_col].values.astype(np.float64)
    if not np.isfinite(atr_vals).any() or np.nanstd(atr_vals) < 1e-12:
        atr_vals = pd.Series(train_df[high_col].values - train_df[low_col].values).rolling(14, min_periods=1).mean().values.astype(np.float64)
else:
    atr_vals = pd.Series(train_df[high_col].values - train_df[low_col].values).rolling(14, min_periods=1).mean().values.astype(np.float64)
atr_vals = np.nan_to_num(atr_vals, nan=float(np.nanmedian(atr_vals) if np.isfinite(atr_vals).any() else np.nanmedian(np.abs(np.diff(close_prices))) or 0.5))
atr_vals = np.maximum(atr_vals, 1e-4)
up_vols  = train_df[up_vol_col].values.astype(np.float64) if up_vol_col else np.zeros(len(train_df))
dn_vols  = train_df[down_vol_col].values.astype(np.float64) if down_vol_col else np.zeros(len(train_df))

last_zones = []
for i in range(N_train):
    abs_idx = i + lookback_bars
    if i % 5 == 0 or not last_zones:
        lb = min(ZONE_LOOKBACK_PERIOD, abs_idx)
        levels = detect_snr_levels_sequential(price_data_hl, up_to_index=abs_idx, lookback_period=lb, min_distance_pct=ZONE_MIN_DISTANCE_PCT) if abs_idx >= 20 else []
        df_slice = price_data_hl.iloc[max(0, abs_idx - ZONE_LOOKBACK_PERIOD): abs_idx + 1]
        last_zones = create_clustered_zones_sequential(levels, df_slice, n_clusters=min(8, max(3, len(levels)))) if levels else []
    ns, nr = get_nearest_zones(last_zones, close_prices[abs_idx])
    nearest_supp_list[i] = ns
    nearest_res_list[i]  = nr
    if i % 5000 == 0:
        print(f"  zones {i}/{N_train}")
print(f"Zone precompute done in {time.time()-t1:.1f}s")

# ---------- Build static state vectors ----------
print("[Precompute] static state features...")
static_states = np.zeros((N_train, 28), dtype=np.float32)
for i in range(N_train):
    abs_idx = i + lookback_bars
    row = train_df.iloc[abs_idx]
    cp  = close_prices[abs_idx]
    atr = max(0.01, atr_vals[abs_idx])
    bv, sv = up_vols[abs_idx], dn_vols[abs_idx]
    ts = row.get("timestamp", None)
    hour_f, dow, phase = 14.5, 1, "off_hours"
    if ts is not None:
        try:
            ts_pd = pd.Timestamp(ts)
            if ts_pd.tzinfo is None: ts_pd = ts_pd.tz_localize("UTC")
            ts_et = ts_pd.tz_convert("America/New_York")
            hour_f = ts_et.hour + ts_et.minute / 60.0
            dow = ts_et.dayofweek
            if   9.5 <= hour_f < 10.5: phase = "nyse_open"
            elif 15.0 <= hour_f < 16.0: phase = "nyse_power_hour"
            elif 9.5 <= hour_f < 16.0:  phase = "regular_hours"
        except Exception: pass
    sv_vec = meta_strengths[i]
    opt_h = int(np.argmax(sv_vec))
    meta_score = float(sv_vec[opt_h])
    dir_flag = 1.0 if meta_score > 0.5 else (-1.0 if meta_score < 0.5 else 0.0)
    hs = sv_vec.tolist()
    ns = nearest_supp_list[i]
    nr = nearest_res_list[i]
    supp_dist = abs(cp - ns["price_level"]) / cp if ns else 1.0
    res_dist  = abs(cp - nr["price_level"]) / cp if nr else 1.0
    supp_vol_ratio = ns["volume_delta_ratio"] if ns else 0.0
    res_vol_ratio  = nr["volume_delta_ratio"] if nr else 0.0
    total_vol = bv + sv
    vol_delta_ratio = (bv - sv) / (total_vol + 1e-6)
    sin_hour = np.sin(2 * np.pi * hour_f / 24.0)
    cos_hour = np.cos(2 * np.pi * hour_f / 24.0)
    static_states[i] = [
        dir_flag, meta_score, float(meta_rev[i]), float(meta_qmax[i]),
        float(meta_mfe[i]), float(meta_mae[i]),
        hs[0], hs[1], hs[2], hs[3],
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        atr / cp, supp_dist, res_dist,
        supp_vol_ratio, res_vol_ratio, vol_delta_ratio,
        0.0,
        sin_hour, cos_hour, dow / 6.0,
        1.0 if phase == "nyse_open" else 0.0,
        1.0 if phase == "nyse_power_hour" else 0.0,
    ]
static_states = np.nan_to_num(static_states, nan=0.0, posinf=0.0, neginf=0.0)
print("Static state cache ready.")

# ---------- Per-horizon Q-network ----------
# One net, 4 independent heads — each head[h] produces [WAIT, CALL, PUT] for horizon h
# FIX: previous kwargs (input_dim=28) don'"'"'t exist on the dual-input class defined
# in the prior cell — that class requires num_features/ctx_dim/q_lookback. This
# mismatch meant this cell'"'"'s displayed output (if any) was stale from an earlier,
# now-orphaned single-input version of the class, not from this code as it stands.
q_net    = ExecutorQNetwork(num_features=num_features, ctx_dim=28, q_lookback=Q_LOOKBACK, hidden_dim=128, num_horizons=NUM_HORIZONS).to(device)
q_target = ExecutorQNetwork(num_features=num_features, ctx_dim=28, q_lookback=Q_LOOKBACK, hidden_dim=128, num_horizons=NUM_HORIZONS).to(device)
q_target.load_state_dict(q_net.state_dict())
q_opt = optim.AdamW(q_net.parameters(), lr=1e-3, weight_decay=1e-4)

Q_EPOCHS = 30
BATCH_SIZE_Q = 64
BUFFER_CAPACITY = 50000  # used only as fallback; deque maxlen handles the cap
# Separate replay buffer per horizon so each head's gradient is signal-clean
import collections
# Persistent across epochs — do not recreate unless this is the first run
if 'replay_buffers' not in dir() or not isinstance(replay_buffers, list) or not isinstance(replay_buffers[0], collections.deque):
    replay_buffers = [collections.deque(maxlen=50000) for _ in range(NUM_HORIZONS)]
    print("Initialized persistent replay buffers (deque, maxlen=50000)")
else:
    print(f"Reusing existing replay buffers: sizes = {[len(b) for b in replay_buffers]}")

def _batch_from_index_replay(buf_batch, matrix, q_lookback, device):
    """Rebuild feature windows from stored absolute indices (RAM-safe)."""
    fw = np.stack([build_feat_window(matrix, int(b[0]), q_lookback) for b in buf_batch])
    st = np.stack([np.asarray(b[1], dtype=np.float32) for b in buf_batch])
    act = torch.tensor([int(b[2]) for b in buf_batch], dtype=torch.long, device=device).unsqueeze(1)
    rew = torch.tensor([float(b[3]) for b in buf_batch], dtype=torch.float32, device=device).unsqueeze(1)
    nfw = np.stack([build_feat_window(matrix, int(b[4]), q_lookback) for b in buf_batch])
    nst = np.stack([np.asarray(b[5], dtype=np.float32) for b in buf_batch])
    fw_t = torch.tensor(fw, dtype=torch.float32, device=device)
    st_t = torch.tensor(st, dtype=torch.float32, device=device)
    nfw_t = torch.tensor(nfw, dtype=torch.float32, device=device)
    nst_t = torch.tensor(nst, dtype=torch.float32, device=device)
    return fw_t, st_t, act, rew, nfw_t, nst_t



epsilon = 1.0
epsilon_min = 0.05
# Step-level decay: reach epsilon_min by ~halfway through total training steps
_total_q_steps = Q_EPOCHS * N_train
_epsilon_decay_step = (epsilon_min / epsilon) ** (1.0 / max(_total_q_steps * 0.5, 1))
print(f"Epsilon step-decay factor: {_epsilon_decay_step:.8f} (reaches {epsilon_min} at step {int(_total_q_steps * 0.5)})")

mask_engine = HardActionMask()

print(f"[Phase 2] Per-Horizon Q-Learning (4 heads x {Q_EPOCHS} epochs)...")
print(f"  {'Epoch':>5} | {'Loss':>10} | {'eps':>5} | {'5m WAIT/CALL/PUT':>18} | {'15m WAIT/CALL/PUT':>18} | {'30m WAIT/CALL/PUT':>18} | {'1h WAIT/CALL/PUT':>18}")
print(f"  {'-'*105}")

for q_epoch in range(Q_EPOCHS):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Per-horizon position tracking
    open_positions = {h: None for h in range(NUM_HORIZONS)}  # h -> {action, entry_price, entry_i}
    win_streaks    = {h: 0 for h in range(NUM_HORIZONS)}
    loss_streaks   = {h: 0 for h in range(NUM_HORIZONS)}
    max_win_streak = {h: 0 for h in range(NUM_HORIZONS)}
    max_loss_streak = {h: 0 for h in range(NUM_HORIZONS)}
    action_counts  = {h: {H_WAIT: 0, H_CALL: 0, H_PUT: 0} for h in range(NUM_HORIZONS)}

    _q_loss_acc = 0.0
    _q_steps    = 0
    _q_loss_window = 0.0
    _q_steps_window = 0
    # Per-horizon detailed buy/sell (CALL/PUT) settlement + reward tracking
    call_wins   = {h: 0 for h in range(NUM_HORIZONS)}
    call_losses = {h: 0 for h in range(NUM_HORIZONS)}
    put_wins    = {h: 0 for h in range(NUM_HORIZONS)}
    put_losses  = {h: 0 for h in range(NUM_HORIZONS)}
    reward_sum   = {h: {H_WAIT: 0.0, H_CALL: 0.0, H_PUT: 0.0} for h in range(NUM_HORIZONS)}
    reward_count = {h: {H_WAIT: 0,   H_CALL: 0,   H_PUT: 0}   for h in range(NUM_HORIZONS)}
    outcome_seq = {h: [] for h in range(NUM_HORIZONS)}       # True=win, False=loss
    settle_q_wins = {h: [] for h in range(NUM_HORIZONS)}
    settle_q_losses = {h: [] for h in range(NUM_HORIZONS)}

    _LOG_EVERY = max(500, N_train // 8)
    print(f"  --- Epoch {q_epoch+1}/{Q_EPOCHS} start | log every {_LOG_EVERY} bars | eps={epsilon:.3f} ---")

    for i in range(N_train):
        abs_idx = i + lookback_bars
        cp  = close_prices[abs_idx]
        atr = max(0.01, atr_vals[abs_idx])
        bv, sv_v = up_vols[abs_idx], dn_vols[abs_idx]
        ns, nr = nearest_supp_list[i], nearest_res_list[i]
        # Dual-input fix: build the raw indicator window once per bar (shared across
        # all 4 horizons this step) — the network needs this alongside the 28-dim
        # context; previously this was never built at all in this cell.
        feat_w = build_feat_window(train_num_matrix, abs_idx, Q_LOOKBACK)

        # --- Process all 4 horizons independently at each bar ---
        for h in range(NUM_HORIZONS):
            lookahead = HORIZON_BARS_LIST[h]

            # Auto-expire position for this horizon
            if open_positions[h] is not None:
                bars_held = i - open_positions[h]["entry_i"]
                if bars_held >= open_positions[h]["horizon"]:
                    entry_p = open_positions[h]["entry_price"]
                    pnl = (cp - entry_p) / (entry_p + 1e-8)
                    if open_positions[h]["action"] == H_PUT:
                        pnl = -pnl
                    if pnl > 0:
                        win_streaks[h] += 1
                        loss_streaks[h] = 0
                        max_win_streak[h] = max(max_win_streak[h], win_streaks[h])
                        outcome_seq[h].append(True)
                        if open_positions[h]["action"] == H_CALL: call_wins[h] += 1
                        else: put_wins[h] += 1
                        if "entry_sig_q" in open_positions[h]:
                            settle_q_wins[h].append(float(open_positions[h]["entry_sig_q"]))
                    else:
                        loss_streaks[h] += 1
                        win_streaks[h] = 0
                        max_loss_streak[h] = max(max_loss_streak[h], loss_streaks[h])
                        outcome_seq[h].append(False)
                        if open_positions[h]["action"] == H_CALL: call_losses[h] += 1
                        else: put_losses[h] += 1
                        if "entry_sig_q" in open_positions[h]:
                            settle_q_losses[h].append(float(open_positions[h]["entry_sig_q"]))
                    settle_reward = float(np.clip(pnl - 0.0005, -0.05, 0.05))
                    # Synthetic CLOSE transition into replay buffer for this horizon
                    sc = static_states[i].copy()
                    sc[12] = float(pnl)  # unrealized → realized
                    next_flat = static_states[min(i + 1, N_train - 1)].copy()
                    next_flat[12] = 0.0
                    next_i_settle = min(i + 1, N_train - 1)
                    nfw = build_feat_window(train_num_matrix, next_i_settle + lookback_bars, Q_LOOKBACK)
                    replay_buffers[h].append((abs_idx, sc.copy() if hasattr(sc, "copy") else sc, H_WAIT, float(settle_reward), next_i_settle + lookback_bars, next_flat.copy() if hasattr(next_flat, "copy") else next_flat))
                    open_positions[h] = None

            # Mark-to-market live unrealized PnL
            if open_positions[h] is not None:
                unreal = (cp - open_positions[h]["entry_price"]) / (open_positions[h]["entry_price"] + 1e-8)
                if open_positions[h]["action"] == H_PUT:
                    unreal = -unreal
            else:
                unreal = 0.0

            has_open = open_positions[h] is not None

            # Per-horizon mask: only WAIT allowed while position is open
            if has_open:
                h_mask = np.array([1, 0, 0], dtype=np.int32)  # WAIT only
            else:
                base_mask = mask_engine.get_action_mask(cp, atr, ns, nr, bv, sv_v, has_open_position=False)
                h_mask = np.array([base_mask[0], base_mask[1], base_mask[2]], dtype=np.int32)

            # Build state (inject horizon index as a feature override in slot 15)
            state = static_states[i].copy()
            state[11] = 1.0 if has_open else 0.0
            state[12] = float(unreal)
            state[13] = win_streaks[h] / 10.0
            state[14] = loss_streaks[h] / 10.0
            state[15] = float(h) / 3.0  # horizon identity slot

            valid = [a for a in range(3) if h_mask[a] == 1] or [H_WAIT]

            if random.random() < epsilon:
                action = random.choice(valid)
            else:
                q_net.eval()
                with torch.no_grad():
                    fw_t = torch.tensor(feat_w[None, ...], dtype=torch.float32, device=device)
                    st_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
                    logits = q_net(fw_t, st_t, horizon_idx=h).squeeze(0).cpu().numpy()
                    masked = np.where(h_mask == 1, logits, -1e9)
                    action = int(np.argmax(masked))
            action_counts[h][action] += 1

            if abs_idx + lookahead >= len(train_df):
                continue
            expiry_cp = close_prices[abs_idx + lookahead]
            fwd_pct = float(np.clip((expiry_cp - cp) / (cp + 1e-8), -0.05, 0.05))

            if not has_open and action == H_CALL:
                open_positions[h] = {"action": H_CALL, "entry_price": cp, "entry_i": i, "horizon": lookahead}
            elif not has_open and action == H_PUT:
                open_positions[h] = {"action": H_PUT, "entry_price": cp, "entry_i": i, "horizon": lookahead}

            # ------------------------------------------------------------------
            # Base reward (continuous, same as before)
            # ------------------------------------------------------------------
            if action == H_CALL:
                base_reward = fwd_pct - 0.0005
            elif action == H_PUT:
                base_reward = -fwd_pct - 0.0005
            else:
                # WAIT reward redesign (unchanged logic)
                if h_mask[H_CALL] == 0 and h_mask[H_PUT] == 0:
                    base_reward = 0.0
                else:
                    h_strength = float(meta_strengths[i][h])
                    if h_strength >= 0.58 and abs(fwd_pct) >= 0.001:
                        base_reward = -0.5 * abs(fwd_pct)
                    elif abs(fwd_pct) < 0.0005:
                        base_reward = 0.0002
                    else:
                        base_reward = 0.0
            base_reward = float(np.clip(base_reward, -0.05, 0.05))

            # ------------------------------------------------------------------
            # Signal-quality shaping  (THE GAP BRIDGE)
            # ------------------------------------------------------------------
            # Pull zone / volume / direction from the static state we just built
            _dir_flag   = float(state[0])
            _supp_dist  = float(state[17]) if len(state) > 17 else 1.0
            _res_dist   = float(state[18]) if len(state) > 18 else 1.0
            _vol_delta  = float(state[21]) if len(state) > 21 else 0.0
            _meta_h     = float(meta_strengths[i][h])
            _row        = train_df.iloc[abs_idx]

            sig_q = compute_signal_quality_score(
                action=action,
                meta_strength_h=_meta_h,
                dir_flag=_dir_flag,
                supp_dist=_supp_dist,
                res_dist=_res_dist,
                vol_delta_ratio=_vol_delta,
                row=_row,
            )
            reward = compute_shaped_reward(base_reward, sig_q)
            # running average of quality for the epoch print
            if not hasattr(compute_shaped_reward, "_q_sum"):
                compute_shaped_reward._q_sum = [0.0] * NUM_HORIZONS
                compute_shaped_reward._q_cnt = [0] * NUM_HORIZONS
            compute_shaped_reward._q_sum[h] += sig_q
            compute_shaped_reward._q_cnt[h] += 1
            # Attach quality at entry for settle-time quality-vs-outcome stats
            if open_positions[h] is not None and open_positions[h].get("entry_i") == i:
                open_positions[h]["entry_sig_q"] = float(sig_q)

            next_i = min(i + 1, N_train - 1)
            next_state = static_states[next_i].copy()
            next_state[11] = 1.0 if open_positions[h] is not None else 0.0
            next_state[12] = 0.0
            next_state[13] = win_streaks[h] / 10.0
            next_state[14] = loss_streaks[h] / 10.0
            next_state[15] = float(h) / 3.0

            reward_sum[h][action] += reward
            reward_count[h][action] += 1

            next_fw = build_feat_window(train_num_matrix, next_i + lookback_bars, Q_LOOKBACK)
            replay_buffers[h].append((abs_idx, state.copy(), int(action), float(reward), next_i + lookback_bars, next_state.copy()))

        # Step-level epsilon decay (once per bar across all horizons)
        if epsilon > epsilon_min:
            epsilon = max(epsilon_min, epsilon * _epsilon_decay_step)

        # --- Batch update: train each head from its own buffer ---
        if i % 4 == 0:
            for h in range(NUM_HORIZONS):
                if len(replay_buffers[h]) < BATCH_SIZE_Q:
                    continue
                q_net.train()
                # Oversample CALL/PUT transitions (action in {1,2}) at 3:1 vs WAIT
                # to counteract the WAIT-domination in the buffer
                _buf_list = list(replay_buffers[h])
                _trade_trans = [t for t in _buf_list if int(t[2]) in (H_CALL, H_PUT)]
                _wait_trans  = [t for t in _buf_list if int(t[2]) == H_WAIT]
                if len(_trade_trans) >= 16 and len(_wait_trans) >= 16:
                    n_trade = min(len(_trade_trans), BATCH_SIZE_Q * 3 // 4)
                    n_wait  = BATCH_SIZE_Q - n_trade
                    batch = random.sample(_trade_trans, n_trade) + random.sample(_wait_trans, n_wait)
                    random.shuffle(batch)
                else:
                    batch = random.sample(_buf_list, min(BATCH_SIZE_Q, len(_buf_list)))
                # Tuple layout: (feat_w, ctx_state, action, reward, next_feat_w, next_ctx_state)
                fw_t, st_t, act_b, rew_b, nfw_t, nst_t = _batch_from_index_replay(batch, train_num_matrix, Q_LOOKBACK, device)
                                                                                
                fw_t   = torch.nan_to_num(fw_t, nan=0.0)
                st_t   = torch.nan_to_num(st_t, nan=0.0)
                nfw_t  = torch.nan_to_num(nfw_t, nan=0.0)
                nst_t = torch.nan_to_num(nst_t, nan=0.0)
                rew_b  = torch.nan_to_num(rew_b, nan=0.0)

                # Q-values from head[h] only
                q_vals_b = q_net(fw_t, st_t, horizon_idx=h).gather(1, act_b)

                with torch.no_grad():
                    next_logits = q_target(nfw_t, nst_t, horizon_idx=h)
                    next_q_targ = next_logits.max(dim=1, keepdim=True).values
                    target_q = rew_b + 0.99 * next_q_targ
                    target_q = torch.nan_to_num(target_q, nan=0.0, posinf=1.0, neginf=-1.0)

                loss = nn.MSELoss()(q_vals_b, target_q)
                if torch.isnan(loss):
                    continue
                q_opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(q_net.parameters(), 1.0)
                q_opt.step()

                with torch.no_grad():
                    for tp, p in zip(q_target.parameters(), q_net.parameters()):
                        tp.data.copy_(0.005 * p.data + 0.995 * tp.data)

                _q_loss_acc += loss.item()
                _q_steps += 1
                _q_loss_window += loss.item()
                _q_steps_window += 1

        # ---- Mid-epoch progress (trades, WR/LR, streaks) ----
        if (i + 1) % _LOG_EVERY == 0 or (i + 1) == N_train:
            _pct = 100.0 * (i + 1) / max(N_train, 1)
            _wloss = _q_loss_window / max(_q_steps_window, 1)
            _q_loss_window = 0.0
            _q_steps_window = 0
            parts = []
            for _h in range(NUM_HORIZONS):
                _cw = call_wins[_h]; _cl = call_losses[_h]
                _pw = put_wins[_h]; _pl = put_losses[_h]
                _tr = _cw + _cl + _pw + _pl
                _wins = _cw + _pw
                _wr = (100.0 * _wins / _tr) if _tr > 0 else 0.0
                _lr = (100.0 - _wr) if _tr > 0 else 0.0
                parts.append(
                    f"{HORIZON_LABELS[_h]} tr={_tr} WR={_wr:.1f}% LR={_lr:.1f}% "
                    f"maxW={max_win_streak[_h]} maxL={max_loss_streak[_h]} "
                    f"curW={win_streaks[_h]} curL={loss_streaks[_h]}"
                )
            print(
                f"    [{q_epoch+1}/{Q_EPOCHS} {_pct:5.1f}% bar {i+1}/{N_train}] "
                f"eps={epsilon:.3f} loss={_wloss:.4e} | " + " || ".join(parts)
            )

    # epsilon is now decayed per-step inside the inner loop; just enforce floor here
    epsilon = max(epsilon_min, epsilon)
    avg_l = _q_loss_acc / max(_q_steps, 1)
    ac_strs = " | ".join(
        f"{HORIZON_LABELS[h]} {action_counts[h][H_WAIT]}/{action_counts[h][H_CALL]}/{action_counts[h][H_PUT]}"
        for h in range(NUM_HORIZONS)
    )
    # average signal quality observed this epoch
    q_strs = " | ".join(
        f"{HORIZON_LABELS[h]} q={compute_shaped_reward._q_sum[h]/max(compute_shaped_reward._q_cnt[h],1):.2f}"
        for h in range(NUM_HORIZONS)
    ) if hasattr(compute_shaped_reward, "_q_sum") else ""
    # reset accumulators
    if hasattr(compute_shaped_reward, "_q_sum"):
        compute_shaped_reward._q_sum = [0.0] * NUM_HORIZONS
        compute_shaped_reward._q_cnt = [0] * NUM_HORIZONS
    print(f"  {q_epoch+1:>5} | {avg_l:.4e} | {epsilon:.3f} | {ac_strs}")
    if q_strs:
        print(f"         signal-quality avg: {q_strs}")

    # In-depth per-horizon settlement + consecutive W/L (money-mgmt inputs)
    print(f"  {'─'*100}")
    for h in range(NUM_HORIZONS):
        c_tot = call_wins[h] + call_losses[h]
        p_tot = put_wins[h] + put_losses[h]
        tot = c_tot + p_tot
        c_wr = (100.0 * call_wins[h] / c_tot) if c_tot > 0 else 0.0
        p_wr = (100.0 * put_wins[h] / p_tot) if p_tot > 0 else 0.0
        overall_wr = (100.0 * (call_wins[h] + put_wins[h]) / tot) if tot > 0 else 0.0
        overall_lr = (100.0 - overall_wr) if tot > 0 else 0.0
        avg_r = {a: (reward_sum[h][a] / reward_count[h][a] if reward_count[h][a] > 0 else 0.0)
                 for a in (H_WAIT, H_CALL, H_PUT)}
        seq = outcome_seq[h]
        avg_w_run = avg_l_run = 0.0
        n_w_runs = n_l_runs = 0
        if seq:
            run_len = 1
            runs_w, runs_l = [], []
            for k in range(1, len(seq)):
                if seq[k] == seq[k - 1]:
                    run_len += 1
                else:
                    (runs_w if seq[k - 1] else runs_l).append(run_len)
                    run_len = 1
            (runs_w if seq[-1] else runs_l).append(run_len)
            n_w_runs, n_l_runs = len(runs_w), len(runs_l)
            avg_w_run = float(np.mean(runs_w)) if runs_w else 0.0
            avg_l_run = float(np.mean(runs_l)) if runs_l else 0.0
        q_w = float(np.mean(settle_q_wins[h])) if settle_q_wins[h] else float("nan")
        q_l = float(np.mean(settle_q_losses[h])) if settle_q_losses[h] else float("nan")
        print(f"      ↳ {HORIZON_LABELS[h]:>4}: trades={tot}  WR={overall_wr:.1f}%  LR={overall_lr:.1f}%")
        print(f"         CALL W={call_wins[h]}/L={call_losses[h]} ({c_wr:.1f}%) | "
              f"PUT W={put_wins[h]}/L={put_losses[h]} ({p_wr:.1f}%)")
        print(f"         streaks maxW={max_win_streak[h]} maxL={max_loss_streak[h]}  "
              f"avgW_run={avg_w_run:.2f} (n={n_w_runs}) avgL_run={avg_l_run:.2f} (n={n_l_runs})  "
              f"final={'W'+str(win_streaks[h]) if win_streaks[h] else 'L'+str(loss_streaks[h])}")
        print(f"         avg reward WAIT={avg_r[H_WAIT]:+.5f} CALL={avg_r[H_CALL]:+.5f} PUT={avg_r[H_PUT]:+.5f}  "
              f"entry_q wins={q_w:.3f} losses={q_l:.3f}")
    print(f"  {'─'*100}")

print("Per-Horizon Q-Executor Training Complete.")



In [12]:
# =============================================================================
# PHASE 3 & 4: OOS EVAL — STREAKS + MARTINGALE (max 4) + STRENGTH DIAGNOSTICS
# =============================================================================

# ── precompute (unchanged logic) ─────────────────────────────────────────────
print("\n" + "=" * 92)
print("PRECOMPUTING OUT-OF-SAMPLE TEST STATE VECTORS & ZONES")
print("=" * 92)

test_matrix = np.nan_to_num(test_df[feature_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
N_test = len(test_df) - lookback_bars - 12
q_net.eval()
net.eval()

test_meta_strengths = np.zeros((N_test, 4), dtype=np.float32)
test_meta_qmax = np.zeros(N_test, dtype=np.float32)
test_meta_rev = np.zeros(N_test, dtype=np.float32)
test_meta_mfe = np.zeros(N_test, dtype=np.float32)
test_meta_mae = np.zeros(N_test, dtype=np.float32)

with torch.no_grad():
    for start in range(0, N_test, 256):
        end = min(start + 256, N_test)
        batch_x = np.stack([test_matrix[i: i + lookback_bars].flatten() for i in range(start, end)])
        x_t = torch.tensor(batch_x, dtype=torch.float32, device=device)
        q_vals, strength, pips, risk, liq, rev = net(x_t)
        test_meta_strengths[start:end] = strength.cpu().numpy()
        test_meta_qmax[start:end] = q_vals.max(dim=1).values.cpu().numpy()
        test_meta_rev[start:end] = rev.squeeze(-1).cpu().numpy() if rev.ndim > 1 else rev.cpu().numpy()
        if risk.shape[-1] >= 2:
            test_meta_mfe[start:end] = risk[:, 0].cpu().numpy()
            test_meta_mae[start:end] = risk[:, 1].cpu().numpy()

test_meta_strengths = np.nan_to_num(test_meta_strengths, nan=0.5)
test_meta_qmax = np.nan_to_num(test_meta_qmax, nan=0.5)
test_meta_rev = np.nan_to_num(test_meta_rev, nan=0.2)
test_meta_mfe = np.nan_to_num(test_meta_mfe, nan=0.5)
test_meta_mae = np.nan_to_num(test_meta_mae, nan=0.15)

test_price_data_hl = test_df[[open_col, high_col, low_col, close_col, vol_col]].rename(
    columns={open_col: "Open", high_col: "High", low_col: "Low", close_col: "Close", vol_col: "Volume"})
test_close_prices = test_df[close_col].values.astype(np.float64)
test_atr_vals = test_df[atr_col].values.astype(np.float64) if atr_col else test_close_prices * 0.005
test_up_vols = test_df[up_vol_col].values.astype(np.float64) if up_vol_col else np.zeros(len(test_df))
test_dn_vols = test_df[down_vol_col].values.astype(np.float64) if down_vol_col else np.zeros(len(test_df))

test_nearest_supp = [None] * N_test
test_nearest_res = [None] * N_test
last_test_zones = []
for i in range(N_test):
    abs_idx = i + lookback_bars
    if i % 5 == 0 or not last_test_zones:
        lb = min(ZONE_LOOKBACK_PERIOD, abs_idx)
        levels = detect_snr_levels_sequential(
            test_price_data_hl, up_to_index=abs_idx, lookback_period=lb,
            min_distance_pct=ZONE_MIN_DISTANCE_PCT) if abs_idx >= 20 else []
        df_slice = test_price_data_hl.iloc[max(0, abs_idx - ZONE_LOOKBACK_PERIOD): abs_idx + 1]
        last_test_zones = create_clustered_zones_sequential(
            levels, df_slice, n_clusters=min(8, max(3, len(levels)))) if levels else []
    ns, nr = get_nearest_zones(last_test_zones, test_close_prices[abs_idx])
    test_nearest_supp[i] = ns
    test_nearest_res[i] = nr

test_static_states = np.zeros((N_test, 28), dtype=np.float32)
for i in range(N_test):
    abs_idx = i + lookback_bars
    row = test_df.iloc[abs_idx]
    cp = test_close_prices[abs_idx]
    atr = max(0.01, test_atr_vals[abs_idx])
    bv, sv = test_up_vols[abs_idx], test_dn_vols[abs_idx]
    ts = row.get("timestamp", None)
    hour_f, dow, phase = 14.5, 1, "off_hours"
    if ts is not None:
        try:
            ts_pd = pd.Timestamp(ts)
            if ts_pd.tzinfo is None:
                ts_pd = ts_pd.tz_localize("UTC")
            ts_et = ts_pd.tz_convert("America/New_York")
            hour_f = ts_et.hour + ts_et.minute / 60.0
            dow = ts_et.dayofweek
            if 9.5 <= hour_f < 10.5:
                phase = "nyse_open"
            elif 15.0 <= hour_f < 16.0:
                phase = "nyse_power_hour"
            elif 9.5 <= hour_f < 16.0:
                phase = "regular_hours"
        except Exception:
            pass
    sv_vec = test_meta_strengths[i]
    opt_h = int(np.argmax(sv_vec))
    meta_score = float(sv_vec[opt_h])
    dir_flag = 1.0 if meta_score > 0.5 else (-1.0 if meta_score < 0.5 else 0.0)
    hs = sv_vec.tolist()
    ns, nr = test_nearest_supp[i], test_nearest_res[i]
    supp_dist = abs(cp - ns["price_level"]) / cp if ns else 1.0
    res_dist = abs(cp - nr["price_level"]) / cp if nr else 1.0
    supp_vol_ratio = ns["volume_delta_ratio"] if ns else 0.0
    res_vol_ratio = nr["volume_delta_ratio"] if nr else 0.0
    total_vol = bv + sv
    vol_delta_ratio = (bv - sv) / (total_vol + 1e-6)
    sin_hour = np.sin(2 * np.pi * hour_f / 24.0)
    cos_hour = np.cos(2 * np.pi * hour_f / 24.0)
    test_static_states[i] = [
        dir_flag, meta_score, float(test_meta_rev[i]), float(test_meta_qmax[i]),
        float(test_meta_mfe[i]), float(test_meta_mae[i]),
        hs[0], hs[1], hs[2], hs[3],
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        atr / cp, supp_dist, res_dist,
        supp_vol_ratio, res_vol_ratio, vol_delta_ratio,
        0.0, sin_hour, cos_hour, dow / 6.0,
        1.0 if phase == "nyse_open" else 0.0,
        1.0 if phase == "nyse_power_hour" else 0.0,
    ]
test_static_states = np.nan_to_num(test_static_states, nan=0.0, posinf=0.0, neginf=0.0)

# =============================================================================
# PHASE 3 & 4 PATCH — STREAKS + MONEY MGMT + MARTINGALE (max 4 steps)
#
# Changes vs prior eval:
#   1. Strength distribution diagnostics (why 3a/4 take 0 trades)
#   2. Dual gate: STRICT (0.60/0.05) + DIAGNOSTIC (0.52/0.02)
#   3. Full streak / rolling WR / Kelly / DD (unchanged helpers)
#   4. NEW: Martingale after loss, multiplier x2, HARD CAP at 4 steps
#      (reset to base size on win or after 4 consecutive losses)
#   5. Equity curves under flat 1R vs martingale for money-mgmt choice
# =============================================================================
import os
import torch
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPIRY_HORIZONS    = {"5m (1 bar)": 1, "15m (3 bars)": 3, "30m (6 bars)": 6, "1h (12 bars)": 12}
HORIZON_BARS_LIST  = list(EXPIRY_HORIZONS.values())
HORIZON_LABELS     = list(EXPIRY_HORIZONS.keys())
H_WAIT, H_CALL, H_PUT = 0, 1, 2

# Gates
CONFIDENCE_THRESHOLD = 0.60
HORIZON_MARGIN       = 0.05
DIAG_CONFIDENCE      = 0.52   # diagnostic only — not for live until meta calibrates
DIAG_MARGIN          = 0.02

# Martingale
BASE_RISK_R          = 1.0    # base risk units per trade
MARTINGALE_MULT      = 2.0    # size *= MULT after each loss
MARTINGALE_MAX_STEPS = 4      # after 4 losses in a row, hard reset to base
ROLL_WINDOWS         = (10, 20, 50)


# ── helpers ──────────────────────────────────────────────────────────────────
def streak_stats(outcomes):
    if not outcomes:
        return dict(max_w=0, max_l=0, avg_w=0.0, avg_l=0.0,
                    n_w_streaks=0, n_l_streaks=0, win_streaks=[], loss_streaks=[],
                    final_streak=0, final_is_win=None)
    win_streaks, loss_streaks = [], []
    cw = cl = 0
    last = None
    for o in outcomes:
        if o == 1:
            if last == 0 and cl:
                loss_streaks.append(cl)
            cw += 1; cl = 0; last = 1
        else:
            if last == 1 and cw:
                win_streaks.append(cw)
            cl += 1; cw = 0; last = 0
    if cw: win_streaks.append(cw)
    if cl: loss_streaks.append(cl)
    return dict(
        max_w=max(win_streaks) if win_streaks else 0,
        max_l=max(loss_streaks) if loss_streaks else 0,
        avg_w=float(np.mean(win_streaks)) if win_streaks else 0.0,
        avg_l=float(np.mean(loss_streaks)) if loss_streaks else 0.0,
        n_w_streaks=len(win_streaks), n_l_streaks=len(loss_streaks),
        win_streaks=win_streaks, loss_streaks=loss_streaks,
        final_streak=cw if last == 1 else cl,
        final_is_win=(last == 1) if last is not None else None,
    )


def rolling_wr(outcomes, window):
    out = []
    for i in range(len(outcomes)):
        if i + 1 < window:
            out.append(np.nan)
        else:
            chunk = outcomes[i + 1 - window: i + 1]
            out.append(100.0 * sum(chunk) / window)
    return out


def simulate_flat(outcomes, risk_r=BASE_RISK_R):
    """+risk on win, -risk on loss. Returns equity series (starts 0)."""
    eq = [0.0]
    for o in outcomes:
        eq.append(eq[-1] + (risk_r if o else -risk_r))
    return np.array(eq)


def simulate_martingale(outcomes, base_r=BASE_RISK_R, mult=MARTINGALE_MULT, max_steps=MARTINGALE_MAX_STEPS):
    """
    After a loss: size *= mult, consecutive_loss_count += 1.
    After a win OR when consecutive_loss_count hits max_steps: reset size to base_r.
    Cap ensures the 5th loss in a row is NOT larger — sequence of sizes for 4 losses:
      base, base*mult, base*mult^2, base*mult^3  then reset.
    With mult=2, max_steps=4: risks = 1, 2, 4, 8  (max exposure 8R on 4th loss).
    """
    eq = [0.0]
    size = base_r
    loss_streak = 0
    sizes_used = []
    for o in outcomes:
        sizes_used.append(size)
        if o:
            eq.append(eq[-1] + size)
            size = base_r
            loss_streak = 0
        else:
            eq.append(eq[-1] - size)
            loss_streak += 1
            if loss_streak >= max_steps:
                size = base_r
                loss_streak = 0
            else:
                size = size * mult
    return np.array(eq), sizes_used


def equity_stats(eq):
    if len(eq) < 2:
        return dict(final=0.0, max_dd=0.0, max_dd_pct=0.0, peak=0.0)
    peak = np.maximum.accumulate(eq)
    dd = peak - eq
    max_dd = float(dd.max())
    # pct vs peak at that point (avoid /0)
    with np.errstate(divide="ignore", invalid="ignore"):
        dd_pct = np.where(peak > 0, dd / peak * 100.0, 0.0)
    return dict(
        final=float(eq[-1]),
        max_dd=max_dd,
        max_dd_pct=float(np.nanmax(dd_pct)) if len(dd_pct) else 0.0,
        peak=float(peak.max()),
    )


def money_mgmt_summary(outcomes, label=""):
    n = len(outcomes)
    if n == 0:
        print(f"  [{label}] no trades")
        return None
    wins = sum(outcomes)
    losses = n - wins
    wr = 100.0 * wins / n
    lr = 100.0 * losses / n
    ss = streak_stats(outcomes)
    edge = (wins - losses) / n
    kelly = max(0.0, edge)

    eq_flat = simulate_flat(outcomes)
    st_flat = equity_stats(eq_flat)
    eq_mart, sizes = simulate_martingale(outcomes)
    st_mart = equity_stats(eq_mart)

    print(f"  [{label}]")
    print(f"    Trades={n}  Wins={wins}  Losses={losses}  WR={wr:.2f}%  LR={lr:.2f}%")
    print(f"    Streaks  maxW={ss['max_w']} maxL={ss['max_l']}  "
          f"avgW={ss['avg_w']:.1f} avgL={ss['avg_l']:.1f}  "
          f"(#{ss['n_w_streaks']} W / #{ss['n_l_streaks']} L streaks)")
    print(f"    Final streak: {ss['final_streak']} "
          f"({'WIN' if ss['final_is_win'] else 'LOSS' if ss['final_is_win'] is False else 'n/a'})")
    print(f"    Flat 1R:   final={st_flat['final']:+.1f}R  maxDD={st_flat['max_dd']:.1f}R  Kelly≈{kelly:.3f}")
    print(f"    Martingale x{MARTINGALE_MULT} max{MARTINGALE_MAX_STEPS}:  "
          f"final={st_mart['final']:+.1f}R  maxDD={st_mart['max_dd']:.1f}R  "
          f"max_size_used={max(sizes) if sizes else 0:.1f}R")
    for w in ROLL_WINDOWS:
        if n >= w:
            rwr = rolling_wr(outcomes, w)
            valid = [x for x in rwr if not np.isnan(x)]
            print(f"    Rolling WR@{w}: last={valid[-1]:.1f}%  min={min(valid):.1f}%  "
                  f"max={max(valid):.1f}%  mean={np.mean(valid):.1f}%")
    return dict(
        outcomes=outcomes, wr=wr, lr=lr, streaks=ss, kelly=kelly,
        flat=st_flat, martingale=st_mart, sizes=sizes,
        eq_flat=eq_flat, eq_mart=eq_mart,
    )


# ── strength diagnostics ─────────────────────────────────────────────────────
print("\n" + "=" * 92)
print("META STRENGTH DIAGNOSTICS (why 3a/4 may take 0 trades)")
print("=" * 92)
s = test_meta_strengths
mx = s.max(axis=1)
margin = np.sort(s, axis=1)[:, -1] - np.sort(s, axis=1)[:, -2]
print(f"  max(strength)  mean={mx.mean():.3f}  std={mx.std():.3f}  "
      f"p50={np.percentile(mx,50):.3f}  p90={np.percentile(mx,90):.3f}  p99={np.percentile(mx,99):.3f}")
print(f"  margin(1st-2nd) p50={np.percentile(margin,50):.3f}  p90={np.percentile(margin,90):.3f}")
for thr in (0.50, 0.52, 0.55, 0.58, 0.60, 0.65):
    print(f"  fraction max>={thr:.2f}: {(mx >= thr).mean()*100:.1f}%")
print(f"  fraction margin>={HORIZON_MARGIN:.2f}: {(margin >= HORIZON_MARGIN).mean()*100:.1f}%")
print(f"  fraction (max>={CONFIDENCE_THRESHOLD} AND margin>={HORIZON_MARGIN}): "
      f"{((mx >= CONFIDENCE_THRESHOLD) & (margin >= HORIZON_MARGIN)).mean()*100:.1f}%")
print(f"  per-horizon mean strength: {s.mean(0)}")


# ── precompute is assumed already done in the notebook (test_* arrays live) ──
# If running this cell standalone after a prior eval cell, test_* exist.
# Re-bind helpers that need q_net / mask.
mask_engine = HardActionMask()

def _get_h_logits(state, abs_idx, h, has_open):
    # FIX: q_net is the dual-input ExecutorQNetwork from Cell 6 —
    # forward(self, feat_window, ctx, horizon_idx=None) requires the raw
    # indicator window as well as the 28-dim context. The previous version
    # only passed the context tensor, which would crash with a missing
    # positional argument the moment this cell actually ran (explains why
    # this cell had zero output — it never got past the first call).
    state = state.copy()
    state[11] = 1.0 if has_open else 0.0
    state[15] = float(h) / 3.0
    feat_w = build_feat_window(test_matrix, abs_idx, Q_LOOKBACK)
    with torch.no_grad():
        fw_t = torch.tensor(feat_w[None, ...], dtype=torch.float32, device=device)
        st_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        return q_net(fw_t, st_t, horizon_idx=h).squeeze(0).cpu().numpy()

def _h_mask(cp, atr, ns, nr, bv, sv, has_open):
    if has_open:
        return np.array([1, 0, 0], dtype=np.int32)
    base = mask_engine.get_action_mask(cp, atr, ns, nr, bv, sv, has_open_position=False)
    return np.array([base[0], base[1], base[2]], dtype=np.int32)

def _pick_action(logits, mask):
    return int(np.argmax(np.where(mask == 1, logits, -1e9)))


def run_recommended(conf_thr, margin_thr, tag):
    outcomes, trade_log = [], []
    wins = losses = waits = skipped = 0
    open_until = -1
    cw = cl = 0
    for idx in range(N_test):
        if idx < open_until:
            continue
        abs_idx = idx + lookback_bars
        sv = test_meta_strengths[idx]
        rec_h = int(np.argmax(sv))
        best_sv = float(sv[rec_h])
        sorted_sv = sorted(sv.tolist(), reverse=True)
        margin_ok = (sorted_sv[0] - sorted_sv[1]) >= margin_thr
        lookahead = HORIZON_BARS_LIST[rec_h]
        if abs_idx + lookahead >= len(test_df):
            continue
        if best_sv < conf_thr or not margin_ok:
            skipped += 1
            waits += 1
            continue
        cp = test_close_prices[abs_idx]
        exp_cp = test_close_prices[abs_idx + lookahead]
        state = test_static_states[idx].copy()
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        action = _pick_action(_get_h_logits(state, abs_idx, rec_h, False), hm)
        if action in (H_CALL, H_PUT):
            open_until = idx + lookahead
            win = int((exp_cp > cp) if action == H_CALL else (exp_cp < cp))
            outcomes.append(win)
            if win:
                wins += 1; cw += 1; cl = 0
            else:
                losses += 1; cl += 1; cw = 0
            trade_log.append({
                "idx": idx, "abs_idx": abs_idx, "horizon": rec_h,
                "lookahead": lookahead,
                "action": "CALL" if action == H_CALL else "PUT",
                "entry": cp, "exit": exp_cp, "win": win,
                "strength": best_sv, "margin": sorted_sv[0] - sorted_sv[1],
                "running_wr": 100.0 * wins / (wins + losses),
                "cur_w_streak": cw, "cur_l_streak": cl,
            })
        else:
            waits += 1
    tot = wins + losses
    wr = 100.0 * wins / tot if tot else 0.0
    print(f"  [{tag}] conf={conf_thr} margin={margin_thr} | "
          f"trades={tot} W={wins} L={losses} waits={waits} skips={skipped} WR={wr:.2f}%")
    mm = money_mgmt_summary(outcomes, label=tag)
    return outcomes, trade_log, mm



# -----------------------------------------------------------------------------
# Signal-quality diagnostic helper (mirrors Phase-2 shaping for OOS analysis)
# -----------------------------------------------------------------------------
def _oos_signal_quality(action, state, meta_strength_h, row=None):
    """Lightweight quality score for trade-log enrichment (same math as training)."""
    try:
        return compute_signal_quality_score(
            action=action,
            meta_strength_h=float(meta_strength_h),
            dir_flag=float(state[0]),
            supp_dist=float(state[17]) if len(state) > 17 else 1.0,
            res_dist=float(state[18]) if len(state) > 18 else 1.0,
            vol_delta_ratio=float(state[21]) if len(state) > 21 else 0.0,
            row=row,
        )
    except NameError:
        # training cell not run / functions not in scope
        return float("nan")

# ── Phase 3a STRICT + DIAGNOSTIC ─────────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 3a: RECOMMENDED HORIZON — STRICT GATE")
print("=" * 92)
outcomes_rec, trade_log_rec, mm_rec = run_recommended(
    CONFIDENCE_THRESHOLD, HORIZON_MARGIN, "3a STRICT")

print("\n" + "=" * 92)
print("PHASE 3a-DIAG: RECOMMENDED HORIZON — RELAXED GATE (analysis only)")
print("=" * 92)
outcomes_diag, trade_log_diag, mm_diag = run_recommended(
    DIAG_CONFIDENCE, DIAG_MARGIN, "3a DIAG")


# ── Phase 3b counterfactual ──────────────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 3b: COUNTERFACTUAL — EACH HEAD FORCED ON ITS OWN HORIZON")
print("=" * 92)
mm_by_horizon = {}
outcomes_by_h = {}
for h_idx, (exp_label, lookahead) in enumerate(EXPIRY_HORIZONS.items()):
    outcomes_h = []
    wins = losses = waits = 0
    cw = cl = mw = ml = 0
    open_until = -1
    for idx in range(N_test):
        if idx < open_until:
            continue
        abs_idx = idx + lookback_bars
        if abs_idx + lookahead >= len(test_df):
            continue
        cp = test_close_prices[abs_idx]
        exp_cp = test_close_prices[abs_idx + lookahead]
        state = test_static_states[idx].copy()
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        action = _pick_action(_get_h_logits(state, abs_idx, h_idx, False), hm)
        if action in (H_CALL, H_PUT):
            open_until = idx + lookahead
            win = int((exp_cp > cp) if action == H_CALL else (exp_cp < cp))
            outcomes_h.append(win)
            if win:
                wins += 1; cw += 1; cl = 0
            else:
                losses += 1; cl += 1; cw = 0
            mw = max(mw, cw); ml = max(ml, cl)
        else:
            waits += 1
    tot = wins + losses
    wr = 100.0 * wins / tot if tot else 0.0
    print(f"  {exp_label:<18} | trades={tot:>5} W={wins:>4} L={losses:>4} waits={waits:>5} "
          f"WR={wr:>6.2f}% | maxW={mw} maxL={ml}")
    mm_by_horizon[exp_label] = money_mgmt_summary(outcomes_h, label=f"3b {exp_label}")
    outcomes_by_h[exp_label] = outcomes_h


# ── Phase 3c baselines ───────────────────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 3c: SANITY BASELINES — Always-CALL / Always-PUT (mask-gated)")
print("=" * 92)
for h_idx, (exp_label, lookahead) in enumerate(EXPIRY_HORIZONS.items()):
    call_w = call_t = put_w = put_t = 0
    open_c = open_p = -1
    for idx in range(N_test):
        abs_idx = idx + lookback_bars
        if abs_idx + lookahead >= len(test_df):
            continue
        cp = test_close_prices[abs_idx]
        exp_cp = test_close_prices[abs_idx + lookahead]
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        if idx >= open_c and hm[H_CALL] == 1:
            call_t += 1; open_c = idx + lookahead
            if exp_cp > cp: call_w += 1
        if idx >= open_p and hm[H_PUT] == 1:
            put_t += 1; open_p = idx + lookahead
            if exp_cp < cp: put_w += 1
    cwr = 100.0 * call_w / call_t if call_t else 0.0
    pwr = 100.0 * put_w / put_t if put_t else 0.0
    print(f"  {exp_label:<18} | CALL {call_t:>4} WR={cwr:>5.1f}% | PUT {put_t:>4} WR={pwr:>5.1f}%")


# ── Phase 4 portfolio (strict gate) ──────────────────────────────────────────
print("\n" + "=" * 92)
print("PHASE 4: MULTI-HORIZON CONCURRENT PORTFOLIO (strict confidence gate)")
print("=" * 92)
active_horizon_until = {h: -1 for h in range(4)}
horizon_outcomes = {h: [] for h in range(4)}
portfolio_outcomes = []
cw_p = cl_p = mw_p = ml_p = 0
for idx in range(N_test):
    abs_idx = idx + lookback_bars
    cp = test_close_prices[abs_idx]
    sv = test_meta_strengths[idx]
    for h in range(4):
        if idx < active_horizon_until[h]:
            continue
        lookahead = HORIZON_BARS_LIST[h]
        if abs_idx + lookahead >= len(test_df):
            continue
        if float(sv[h]) < CONFIDENCE_THRESHOLD:
            continue
        exp_cp = test_close_prices[abs_idx + lookahead]
        state = test_static_states[idx].copy()
        hm = _h_mask(cp, max(0.01, test_atr_vals[abs_idx]),
                     test_nearest_supp[idx], test_nearest_res[idx],
                     test_up_vols[abs_idx], test_dn_vols[abs_idx], False)
        action = _pick_action(_get_h_logits(state, abs_idx, h, False), hm)
        if action in (H_CALL, H_PUT):
            active_horizon_until[h] = idx + lookahead
            outcome = int(exp_cp > cp) if action == H_CALL else int(exp_cp < cp)
            portfolio_outcomes.append(outcome)
            horizon_outcomes[h].append(outcome)
            if outcome:
                cw_p += 1; cl_p = 0
            else:
                cl_p += 1; cw_p = 0
            mw_p = max(mw_p, cw_p); ml_p = max(ml_p, cl_p)

print(f"  PORTFOLIO trades={len(portfolio_outcomes)}  maxW={mw_p} maxL={ml_p}")
mm_port = money_mgmt_summary(portfolio_outcomes, label="4 Portfolio")
for h in range(4):
    money_mgmt_summary(horizon_outcomes[h], label=f"4 slot {HORIZON_LABELS[h]}")


# ── Martingale recommendation block ──────────────────────────────────────────
print("\n" + "=" * 92)
print("MARTINGALE MONEY-MGMT GUIDE (from this OOS path)")
print(f"  Rules: after LOSS size *= {MARTINGALE_MULT}; reset on WIN or after "
      f"{MARTINGALE_MAX_STEPS} consecutive losses")
print(f"  Size ladder: " + " → ".join(
    f"{BASE_RISK_R * (MARTINGALE_MULT ** i):.0f}R" for i in range(MARTINGALE_MAX_STEPS)))
print("=" * 92)
print("""
  Interpretation:
  - avgL ≈ 2.x  →  typical loss run is ~2 trades; maxL is the danger number.
  - Martingale max 4 steps means the 4th loss is at 8R (if mult=2, base=1).
  - If maxL on a horizon >> 4, martingale WILL hit the cap often and still
    leave residual losing streaks at base size — it does NOT remove long runs.
  - Only consider martingale on a horizon where:
        WR > 52%,  maxL is modest,  and flat maxDD is already acceptable.
  - Your 5m path (WR~49%, maxL=15) is a poor martingale candidate.
  - 15m (WR~52.5%, maxL=8) is the least-bad candidate; still size base risk
    so that 8R (4th step) is an acceptable hit (e.g. base 0.25–0.5% equity).
""")


# ── export logs ──────────────────────────────────────────────────────────────
out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
if trade_log_rec:
    pd.DataFrame(trade_log_rec).to_csv(os.path.join(out_dir, "phase3a_strict_trade_log.csv"), index=False)
    print(f"  Saved phase3a_strict_trade_log.csv ({len(trade_log_rec)} rows)")
if trade_log_diag:
    pd.DataFrame(trade_log_diag).to_csv(os.path.join(out_dir, "phase3a_diag_trade_log.csv"), index=False)
    print(f"  Saved phase3a_diag_trade_log.csv ({len(trade_log_diag)} rows)")

# Save equity curves for the best counterfactual horizon if any
for lab, outs in outcomes_by_h.items():
    if not outs:
        continue
    eq_f = simulate_flat(outs)
    eq_m, _ = simulate_martingale(outs)
    df_eq = pd.DataFrame({"flat_R": eq_f, "martingale_R": eq_m})
    safe = lab.replace(" ", "_").replace("(", "").replace(")", "")
    path = os.path.join(out_dir, f"equity_{safe}.csv")
    df_eq.to_csv(path, index=False)
    print(f"  Saved {path}")



PRECOMPUTING OUT-OF-SAMPLE TEST STATE VECTORS & ZONES

META STRENGTH DIAGNOSTICS (why 3a/4 may take 0 trades)
  max(strength)  mean=0.528  std=0.030  p50=0.519  p90=0.548  p99=0.657
  margin(1st-2nd) p50=0.004  p90=0.011
  fraction max>=0.50: 99.5%
  fraction max>=0.52: 47.3%
  fraction max>=0.55: 9.3%
  fraction max>=0.58: 4.9%
  fraction max>=0.60: 3.5%
  fraction max>=0.65: 1.3%
  fraction margin>=0.05: 1.0%
  fraction (max>=0.6 AND margin>=0.05): 1.0%
  per-horizon mean strength: [0.49691352 0.495486   0.5234648  0.52567405]

PHASE 3a: RECOMMENDED HORIZON — STRICT GATE
  [3a STRICT] conf=0.6 margin=0.05 | trades=4 W=2 L=2 waits=7289 skips=7251 WR=50.00%
  [3a STRICT]
    Trades=4  Wins=2  Losses=2  WR=50.00%  LR=50.00%
    Streaks  maxW=1 maxL=1  avgW=1.0 avgL=1.0  (#2 W / #2 L streaks)
    Final streak: 1 (LOSS)
    Flat 1R:   final=+0.0R  maxDD=1.0R  Kelly≈0.000
    Martingale x2.0 max4:  final=+1.0R  maxDD=1.0R  max_size_used=2.0R

PHASE 3a-DIAG: RECOMMENDED HORIZON — RELAXED G

In [ ]:
# =============================================================================
# 💾 EXPORT TRAINED CHECKPOINTS TO ZIP FOR BACKEND HYDRATION
# =============================================================================
def export_all_checkpoints_zip(output_zip_path):
    pt_meta_path = os.path.join(OUTPUT_DIR, 'meta_learner_best.pt')
    pt_q_path    = os.path.join(OUTPUT_DIR, 'q_executor_best.pt')

    torch.save(net.state_dict(), pt_meta_path)
    torch.save(q_net.state_dict(), pt_q_path)

    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(pt_meta_path, arcname='meta_learner_best.pt')
        zipf.write(pt_q_path,    arcname='q_executor_best.pt')

    zip_mb = os.path.getsize(output_zip_path) / (1024 * 1024)
    print(f"✅ Checkpoint export complete: {output_zip_path} ({zip_mb:.2f} MB)")
    print("Ready to copy back to backend app/core/ml/checkpoints/")

export_all_checkpoints_zip(ZIP_EXPORT_PATH)
